# ⚔️ 11 · Support vector machines: margins, support vectors and kernels
### The maximum-margin principle derived line by line, hard and soft margins solved as quadratic programs, the dual and its support vectors checked against scikit-learn, the kernel trick verified numerically — and an honest model selection (C, kernel, γ) on leukaemia expression data

**BiologyToML** · Part B — Supervised learning · Module 11 of 24

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/11_support_vector_machines.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`scipy.optimize`), pandas, Matplotlib; scikit-learn (for checking our solutions and for cross-validation) |
| **Prerequisites** | Modules 03 (vectors, dot products), 04 (derivatives, optimisation, Lagrange multipliers) and 09 (logistic regression, linear decision boundaries). |
| **Estimated time** | 170 min (reading + running + the ✏️ exercises) |
| **02-620 lectures covered** | `007_SVM` slides 1–74 and `007_SVM_extra_slides`; model-selection remark of `015_model-selection_mz_040725` slide 3 — see [`SLIDE_MAP.md`](../SLIDE_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> The Golub et al. (1999) leukaemia expression data (72 patients × 3,571 genes) and the Wisconsin
> breast-cancer biopsies are **real**. The two-marker "tumour" scatter plots, the concentric "circles" and the
> three "cell types" are **SIMULATED** toys designed to make the geometry visible.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** the maximum-margin principle and why an SVM is a discriminative, **non-probabilistic** classifier that depends only on its support vectors.
2. **Derive** the margin $M = 2/\lVert w\rVert$ line by line (slides 23–28) and **verify** every step numerically.
3. **Formulate** the hard- and soft-margin SVMs as quadratic programs, **solve** them with SciPy, and **train** the soft-margin primal by subgradient descent on the hinge loss.
4. **Derive** the dual with Lagrange multipliers, **solve** it, and **identify** support vectors from the multipliers ($\alpha_i = 0$, $0<\alpha_i<C$, $\alpha_i = C$), checking them against scikit-learn.
5. **Verify** the kernel identity $\phi(x)^\top\phi(z) = \kappa(x,z)$ numerically and **apply** polynomial and RBF kernels to data no line can separate.
6. **Select** an SVM's hyper-parameters (C, kernel, γ, preprocessing) by cross-validation on training data only, and **interpret** the result on the Golub leukaemia data.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, course toolkit | 5 min |
| 1 · Linear classifiers so far | logistic regression revisited; many separating lines | 15 min |
| 2 · The maximum-margin idea | margin, support vectors | 5 min |
| 3 · Geometry of the margin | w ⟂ boundary, M = 2/‖w‖, the hard-margin QP solved | 30 min |
| 4 · Soft margin | slack variables, the role of C, hinge loss, subgradient descent | 25 min |
| 5 · Duality and support vectors | Lagrangian, the dual QP, α categories, prediction cost | 30 min |
| 6 · Kernels | φ(x) = (x, x²), the kernel identity, polynomial and RBF SVMs | 25 min |
| 7 · More than two classes | one-vs-all | 5 min |
| 8 · Real data and model selection | Golub ALL/AML: choosing C, kernel and γ honestly | 25 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 5 min |

## 🧫 The biological question

A pathologist receives a tumour of unknown origin. Is it breast, lung, colon, a lymphoma, a melanoma? The answer
decides the treatment, and for metastases of unknown primary site it is often unclear from the microscope. In 2001
Ramaswamy and colleagues measured the expression of **16,063 genes in 218 tumours of 14 cancer types** and trained
one **support vector machine (SVM)** per cancer type to recognise that type against all others (slide 2). Their data
have a shape that is typical of genomics: **far more features than samples** (J ≫ N). Slide 3 lists many more SVM
applications in cancer genomics — subtyping from mRNA, methylation or copy number, drug-sensitivity prediction,
driver-mutation discovery (Huang *et al.* 2018).

Why SVMs? With J ≫ N there are infinitely many hyperplanes that separate the training tumours perfectly, so "fit the
training data" no longer determines the answer. The SVM adds a principle: among all separating boundaries, choose
the one that stays **as far as possible from both classes** — the **maximum margin**. The resulting boundary depends
only on the few samples nearest to it, the **support vectors**, and with **kernels** the same machinery draws curved
boundaries without ever computing the curved features.

In this notebook you will derive all of this from the slides, solve the optimisation problems yourself with SciPy,
and finish on the Golub leukaemia data (acute lymphoblastic vs acute myeloid leukaemia, ALL vs AML), where you will
see that an SVM *does* require model-selection choices — contrary to the "SVM: NA" line of the model-selection
lecture.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.datasets import load_breast_cancer, make_circles
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`load_golub` reads the real leukaemia data from the repository (or downloads it; offline it falls back to a clearly labelled simulated stand-in), and the other helpers standardise features, score classifiers, split data and draw decision regions. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_DATA_URL = "https://raw.githubusercontent.com/xulabs/edu/main/ml/tutorial/BiologyToML/datasets/{name}"


CACHE_DIR = os.environ.get("BIO2ML_CACHE", "course_data")


def fetch_course_file(name: str, timeout: float = 60.0) -> str | None:
    """Return a local path to the course data file ``name``, or ``None`` if it cannot be found.

    Looks in the repository's ``datasets/`` folder first (when the notebook runs inside a
    clone of xulabs/edu), then in the download cache, and finally downloads it from GitHub.
    """
    for base in ("../datasets", "datasets", "ml/tutorial/BiologyToML/datasets", CACHE_DIR):
        path = os.path.join(base, name)
        if os.path.exists(path) and os.path.getsize(path) > 0:
            return path
    dest = os.path.join(CACHE_DIR, name)
    os.makedirs(CACHE_DIR, exist_ok=True)
    try:
        with urllib.request.urlopen(REPO_DATA_URL.format(name=name), timeout=timeout) as r, \
                open(dest + ".part", "wb") as f:
            f.write(r.read())
        os.replace(dest + ".part", dest)
        return dest
    except Exception as e:  # noqa: BLE001 — any network problem → offline fallback
        print(f"⚠️  Could not obtain {name} ({type(e).__name__}); using a simulated stand-in instead.")
        return None


def simulate_golub_like(rng: np.random.Generator) -> dict:
    """Simulated stand-in for the Golub data: 72 samples (47 'ALL', 25 'AML') × 3,571 genes.

    Log10-scale expression with gene-specific means, correlated gene modules and
    60 genes that differ between the classes.  Same keys as ``load_golub``.
    """
    n_all, n_aml, J = 47, 25, 3571
    y = np.r_[np.zeros(n_all, int), np.ones(n_aml, int)]
    base = rng.normal(2.6, 0.45, J)
    modules = rng.normal(0, 1, (len(y), 20))                  # 20 hidden "programmes"
    loading = rng.normal(0, 0.15, (20, J)) * (rng.random((20, J)) < 0.1)
    X = base + modules @ loading + rng.normal(0, 0.25, (len(y), J))
    informative = rng.choice(J, 60, replace=False)
    X[:, informative] += np.outer(y - 0.5, rng.choice([-1, 1], 60) * rng.uniform(0.4, 0.9, 60))
    is_train = np.r_[np.arange(n_all) < 27, np.arange(n_aml) < 11]
    return dict(X=X, y=y, labels=np.where(y == 1, "AML", "ALL"), genes=np.array([f"gene{j:04d}" for j in range(J)]),
                is_train=is_train, sample_id=np.arange(1, len(y) + 1), source="SIMULATED stand-in (offline)")


def load_golub() -> dict:
    """Golub et al. (1999) leukaemia expression data, preprocessed as in Dudoit et al. (2002).

    Returns a dict with ``X`` (72 × 3,571, log10 expression), ``y`` (0 = ALL, 1 = AML),
    ``labels``, ``genes`` (Affymetrix probe IDs), ``is_train`` (the original 38/34 split),
    ``sample_id`` and ``source``.  Falls back to ``simulate_golub_like`` when offline.

    Raw intensities were clipped to [100, 16000] before the log10, so the value 2.0 means
    "at or below the detection limit" (about 20 % of all entries).  Some genes are therefore
    constant within a class — per-class variance estimates need a small floor.
    """
    path = fetch_course_file("golub_leukemia.csv.gz")
    if path is None:
        return simulate_golub_like(np.random.default_rng(1999))
    df = pd.read_csv(path)
    return dict(X=df.iloc[:, 3:].to_numpy(float), y=(df["label"] == "AML").to_numpy(int),
                labels=df["label"].to_numpy(), genes=np.array(df.columns[3:]),
                is_train=(df["golub_split"] == "train").to_numpy(), sample_id=df["sample_id"].to_numpy(),
                source="Golub et al. 1999 (real data)")


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


def train_test_indices(n: int, test_frac: float, rng: np.random.Generator, stratify=None):
    """Random train/test split of ``n`` samples; ``stratify`` keeps class proportions equal."""
    if stratify is None:
        perm = rng.permutation(n)
        n_test = int(round(test_frac * n))
        return np.sort(perm[n_test:]), np.sort(perm[:n_test])
    stratify = np.asarray(stratify)
    train, test = [], []
    for c in np.unique(stratify):
        idx = rng.permutation(np.flatnonzero(stratify == c))
        n_test = int(round(test_frac * len(idx)))
        test += list(idx[:n_test])
        train += list(idx[n_test:])
    return np.sort(train), np.sort(test)


def accuracy(y_true, y_pred) -> float:
    """Fraction of correct predictions."""
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def confusion_counts(y_true, y_pred) -> dict:
    """True/false positives/negatives for binary labels (1 = positive)."""
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    return dict(TP=int(np.sum((y_pred == 1) & (y_true == 1))), FP=int(np.sum((y_pred == 1) & (y_true == 0))),
                FN=int(np.sum((y_pred == 0) & (y_true == 1))), TN=int(np.sum((y_pred == 0) & (y_true == 0))))


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


def plot_decision_regions(predict, X, y, ax=None, resolution: int = 200, title: str = "",
                          class_names=None, pad: float = 0.5):
    """Shade the regions a 2-feature classifier assigns to each class and overlay the data.

    ``predict`` maps an (M, 2) array to M integer labels 0, 1, 2, …
    """
    X, y = np.asarray(X, float), np.asarray(y).astype(int)
    ax = ax or plt.gca()
    x0, x1 = X[:, 0].min() - pad, X[:, 0].max() + pad
    y0, y1 = X[:, 1].min() - pad, X[:, 1].max() + pad
    gx, gy = np.meshgrid(np.linspace(x0, x1, resolution), np.linspace(y0, y1, resolution))
    Z = np.asarray(predict(np.c_[gx.ravel(), gy.ravel()])).astype(int).reshape(gx.shape)
    k = max(int(y.max()), int(Z.max())) + 1
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(CLASS_COLORS[:k])
    ax.contourf(gx, gy, Z, levels=np.arange(k + 1) - 0.5, cmap=cmap, alpha=0.18)
    ax.contour(gx, gy, Z, levels=np.arange(k) + 0.5, colors="k", linewidths=0.8)
    for c in range(k):
        m = y == c
        if m.any():
            ax.scatter(X[m, 0], X[m, 1], s=18, color=CLASS_COLORS[c], edgecolor="k", linewidth=0.3,
                       label=(class_names[c] if class_names is not None else f"class {c}"))
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['fetch_course_file', 'simulate_golub_like', 'load_golub', 'standardize', 'train_test_indices', 'accuracy', 'confusion_counts', 'plot_decision_regions']))

---
## 1 · Linear classifiers so far

> 📍 **Where we are.** Slide 4 recalls the two families of classifiers you know. **Generative** classifiers
> (naïve Bayes, [Module 08](08_knn_naive_bayes.ipynb)) model $P(X, Y) = P(X\mid Y)P(Y)$ and obtain $P(Y\mid X)$ by
> Bayes' rule. **Discriminative** classifiers (logistic regression, [Module 09](09_logistic_regression.ipynb)) model
> $P(Y\mid X)$ directly. The SVM is **another discriminative classifier, but non-probabilistic**: it learns a boundary
> and outputs a side of it, not a probability. It is one of the course's non-probabilistic exceptions (with trees and
> k-means); its probabilistic cousin is logistic regression, and Section 4 shows how close the two are.

**Conventions for this module.** The SVM slides code the two classes as $y = +1$ and $y = -1$ (instead of 1/0),
because then "correct side of the boundary" becomes the single condition $y\,(w^\top x + b) > 0$. A sample is a column
vector $x$ with J entries (features); the slides write $n$ for the number of training samples and $m$ for the number
of features (slide 55) — we keep the course's **N samples × J features**. On the slides, $w'x$ means $w^\top x$.

A linear classifier predicts $\hat y = \operatorname{sign}(w^\top x + b)$: the boundary is the line (plane,
hyperplane) $w^\top x + b = 0$ (slide 6). Let us make a toy: two tumour types measured by two marker genes.

In [ ]:
# ── 1.1 A toy: two tumour types, two marker genes (SIMULATED) ──
def two_tumour_types(rng, n: int = 20, spread: float = 0.6, shift: float = 2.0):
    """SIMULATED expression of two marker genes; type +1 (red) around (+shift, +shift), type −1 (blue) around −shift."""
    X_pos = rng.normal(shift, spread, (n // 2, 2))
    X_neg = rng.normal(-shift, spread, (n // 2, 2))
    return np.vstack([X_pos, X_neg]), np.r_[np.ones(n // 2), -np.ones(n // 2)]


def draw_margin(ax, w, b, X, y, sv=None, title: str = "", lim=(-4.5, 4.5)) -> None:
    """Scatter ±1 data and draw w·x + b = 0 (solid) and = ±1 (dashed); circle the support vectors."""
    ax.scatter(*X[y > 0].T, s=22, color=CLASS_COLORS[1], label="type +1", zorder=3)
    ax.scatter(*X[y < 0].T, s=22, color=CLASS_COLORS[0], label="type −1", zorder=3)
    x1 = np.linspace(*lim, 2)
    for level, style in [(0, "-"), (1, "--"), (-1, "--")]:
        ax.plot(x1, (level - b - w[0] * x1) / w[1], "k" + style, lw=1.2 if level == 0 else 0.8)
    if sv is not None:
        ax.scatter(*X[sv].T, s=150, facecolor="none", edgecolor="k", lw=1.2, label="support vectors", zorder=4)
    ax.set_xlim(*lim); ax.set_ylim(*lim); ax.set_aspect("equal")
    ax.set_xlabel("marker gene 1 (log expression, centred)"); ax.set_ylabel("marker gene 2")
    if title:
        ax.set_title(title)


rng = np.random.default_rng(SEED + 1)
X_toy, y_toy = two_tumour_types(rng)
logreg = LogisticRegression(C=1.0).fit(X_toy, y_toy)
fig, ax = plt.subplots(figsize=(4.4, 4.2))
draw_margin(ax, logreg.coef_[0], logreg.intercept_[0], X_toy, y_toy,
            title="logistic regression boundary\n(dashed: where the log-odds are ±1)")
ax.legend(fontsize=7, loc="upper left"); plt.tight_layout(); plt.show()
print("logistic regression weights w =", logreg.coef_[0].round(3), " b =", logreg.intercept_[0].round(3))

🔍 **What just happened?** Logistic regression found a line that separates the two tumour types. Slide 9 reminds us
*how*: maximum conditional likelihood $\arg\max_w \prod_l P(Y^l\mid X^l, W)$ — every training point contributes a
factor, so **every point influences the boundary**, including points far from it. (Here we used scikit-learn's
default L2 penalty, the MAP version of Module 09, because on separable data the unpenalised MLE does not exist — the
weights would grow without bound.)

Slide 10's observation is the starting point of SVMs: **many lines separate these data perfectly.** Which one is
best? Let us draw some and measure, for each, the **margin** — the width of the widest empty band around the line
that is parallel to it.

In [ ]:
# ── 1.2 Many separating lines — and their margins (slide 10) ──
def geometric_margin(w, b, X, y) -> float:
    """Width of the empty band around the line w·x + b = 0: 2 × the smallest signed distance y(w·x + b)/‖w‖."""
    return 2 * np.min(y * (X @ w + b)) / np.linalg.norm(w)


rng = np.random.default_rng(SEED + 12)
lines = []
for _ in range(5000):                                  # random directions and offsets
    w = rng.normal(size=2); b = rng.normal(0, 1.0)
    if np.all(y_toy * (X_toy @ w + b) > 0):            # keep lines that separate the training data
        lines.append((geometric_margin(w, b, X_toy, y_toy), w, b))
lines.sort(key=lambda t: t[0])
print(f"{len(lines)} of 5,000 random lines separate the data; their margins range from "
      f"{lines[0][0]:.2f} to {lines[-1][0]:.2f}")

fig, ax = plt.subplots(figsize=(4.4, 4.2))
ax.scatter(*X_toy[y_toy > 0].T, s=22, color=CLASS_COLORS[1]); ax.scatter(*X_toy[y_toy < 0].T, s=22, color=CLASS_COLORS[0])
x1 = np.linspace(-4.5, 4.5, 2)
for k in np.linspace(0, len(lines) - 1, 6).astype(int):
    M, w, b = lines[k]
    ax.plot(x1, (-b - w[0] * x1) / w[1], lw=1, label=f"margin {M:.2f}")
ax.set_xlim(-4.5, 4.5); ax.set_ylim(-4.5, 4.5); ax.set_aspect("equal"); ax.legend(fontsize=7)
ax.set_title("all of these separate the training data"); ax.set_xlabel("marker gene 1"); ax.set_ylabel("marker gene 2")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Every drawn line has zero training error, yet their margins differ a lot. A line that
grazes one cluster leaves no room for error: a new tumour slightly off its cluster would fall on the wrong side. A
line in the middle of the widest empty corridor is the most robust choice. Keep the best random margin in mind — in
Section 3 we compute the true maximum.

✏️ **Your turn.** Make the two clouds overlap more (`two_tumour_types(rng, spread=1.5)` in 1.1) and re-run 1.1–1.2.
How many of the 5,000 random lines still separate the data? What should a classifier do when *no* line separates
them? (Section 4 answers.)

---
## 2 · The maximum-margin idea

> 📍 **Where we are.** Section 1 showed the problem (too many perfect lines); slides 11–13 give the SVM's answer.

**Instead of fitting all points, focus on the boundary points.** Draw two parallel lines that (i) are parallel to the
decision boundary, (ii) separate the two classes and (iii) are as far apart as possible; put the decision boundary in
the middle. The distance between the two parallel lines is the **margin** $M$. The training points that touch the
margin lines hold it in place like tent poles: they are the **support vectors** — they "support" the decision
boundary. Move any other point (without crossing the margin) and the boundary does not change.

Slide 15 asks *"what is the loss function of the SVM?"* For logistic regression it was the negative log-likelihood.
For the SVM the answer will be "maximise the margin subject to classifying the training data correctly" (Section 3),
relaxed to "trade margin against violations" (Section 4), which we will rewrite as the **hinge loss** plus a ridge
penalty.

⚠️ **Terminology.** The 02-620 slides define the margin $M$ as the full width of the band between the two margin
lines. Some textbooks (ESL, Murphy) call half of that width the margin. Both conventions give the same classifier.

✏️ **Your turn (pen and paper).** Draw three type +1 tumours at (1, 1), (2, 2), (2, 0) and two type −1 tumours at
(−1, −1), (0, −2). Sketch the maximum-margin line by eye and circle the points you expect to be support vectors.
After Section 3.4, check your sketch with `hard_margin_primal`.

### 📑 Slide connection — `007_SVM`, slides 2–13

- **What the slides show.** SVMs in cancer genomics: one-vs-all multiclass prediction of 14 tumour types from 16,063 genes in 218 tumours (slide 2) and a table of applications (3). Generative vs discriminative classifiers; the SVM is discriminative and non-probabilistic (4). Logistic regression's MLE considers all data points and there are many separating lines (6–10). The max-margin classifier focuses on boundary points; margin lines are parallel to the boundary, separate the classes and are as wide as possible; the points on them are the support vectors (11–13).
- **What this notebook adds.** The ±1 label convention and N × J notation made explicit; logistic regression on a simulated two-marker toy; 5,000 random separating lines with their margins measured, so the problem the SVM solves is visible before any algebra; the half-width/full-width margin convention.
- **Exam tip.** Know what distinguishes generative, discriminative and non-probabilistic classifiers, and be able to point at the support vectors in a drawing.

---
## 3 · The geometry of the margin

> 📍 **Where we are.** We must turn "widest band" into a formula in $w$ and $b$ that a computer can optimise
> (slides 16–31). This needs a little vector geometry.

🧮 **Math refresher — vectors, dot products and lengths** (Module 03). A vector $w = (w_1, \dots, w_J)$ is an arrow.
The **dot product** $w^\top x = \sum_{j=1}^{J} w_j x_j$ multiplies matching entries and adds them. The **length**
(Euclidean norm) is $\lVert w\rVert = \sqrt{w^\top w}$. Two vectors are **perpendicular** (at a right angle) exactly
when their dot product is 0. For any vector, $w^\top w = \lVert w\rVert^2$, i.e. $\sqrt{w^\top w}^{\,2} = w^\top w$
(slide 28's note).

The SVM uses three parallel lines (planes in higher dimension):

$$w^\top x + b = +1 \;(\text{the "+1 plane"}), \qquad w^\top x + b = 0 \;(\text{the decision boundary}), \qquad
  w^\top x + b = -1 \;(\text{the "−1 plane"}).$$

*In words:* the boundary sits where the score $w^\top x + b$ is zero, and the margin lines where the score is $\pm 1$.
Why exactly ±1? Multiplying $w$ and $b$ by any positive number $c$ leaves the decision boundary $w^\top x + b = 0$
where it is but multiplies every score by $c$; we use that freedom to make the score of the closest training points
exactly ±1. This **canonical scaling** is what
makes the margin a function of $\lVert w\rVert$ alone.

### 3.1 Fact 1: $w$ is perpendicular to the planes (slide 24)

Take two points $u$ and $v$ on the +1 plane: $w^\top u + b = 1$ and $w^\top v + b = 1$. Subtracting,

$$w^\top u + b - (w^\top v + b) = 1 - 1 \quad\Longrightarrow\quad w^\top (u - v) = 0 .$$

*In words:* the vector $u - v$ runs *along* the plane, and its dot product with $w$ is zero — so $w$ is at a right
angle to every direction inside the plane. The same argument works for the 0 and −1 planes: $w$ points straight across
all three.

In [ ]:
# ── 3.1 Check Fact 1 numerically in 5 dimensions ──
rng = np.random.default_rng(SEED + 31)
w, b = rng.normal(size=5), 0.7                      # an arbitrary hyperplane in J = 5 dimensions


def point_on_plane(level: float) -> np.ndarray:
    """A random point x with w·x + b = level (start anywhere, then slide along w)."""
    x = rng.normal(size=5)
    return x + (level - b - w @ x) / (w @ w) * w


u, v = point_on_plane(1.0), point_on_plane(1.0)
print("w·u + b =", round(w @ u + b, 12), "  w·v + b =", round(w @ v + b, 12))
print("w·(u − v) =", round(w @ (u - v), 12), " → w is perpendicular to the +1 plane")

### 3.2 Fact 2 and the margin, line by line (slides 25–28)

Let $x^+$ be a point on the +1 plane and $x^-$ the point of the −1 plane **closest** to it. The shortest way from
one plane to the other is straight across, i.e. along $w$ (Fact 1), so

$$x^+ = x^- + \lambda w \qquad\text{for some number } \lambda > 0 .$$

*In words:* start at $x^-$ and walk $\lambda$ "steps" of the vector $w$. Collect what we know (slide 26):
(1) $w^\top x^+ + b = +1$; (2) $w^\top x^- + b = -1$; (3) $x^+ = x^- + \lambda w$; (4) $\lVert x^+ - x^-\rVert = M$.
Now the derivation of slide 27, one step per line:

$$\begin{aligned}
w^\top x^+ + b &= +1 && \text{(1)}\\
w^\top(x^- + \lambda w) + b &= +1 && \text{substitute (3)}\\
\underbrace{w^\top x^- + b}_{=\,-1 \text{ by (2)}} + \lambda\, w^\top w &= +1 && \text{multiply out, regroup}\\
-1 + \lambda\, w^\top w &= +1 && \\
\lambda &= \frac{2}{w^\top w}.
\end{aligned}$$

*In words:* crossing from the −1 plane to the +1 plane must raise the score by exactly 2, and each step of $w$ raises
it by $w^\top w$ — so the crossing takes $2/w^\top w$ steps.

And the margin (slide 28):

$$M = \lVert x^+ - x^-\rVert = \lVert \lambda w\rVert = \lambda\,\lVert w\rVert = \frac{2}{w^\top w}\sqrt{w^\top w}
  = \frac{2}{\sqrt{w^\top w}} = \frac{2}{\lVert w\rVert}.$$

*In words:* the width of the band is 2 divided by the length of $w$. **A shorter weight vector means a wider
margin.** Let us check every line with numbers.

In [ ]:
# ── 3.2 Verify λ = 2/wᵀw and M = 2/‖w‖ numerically ──
x_minus = point_on_plane(-1.0)
lam = 2 / (w @ w)
x_plus = x_minus + lam * w
print(f"w·x⁻ + b = {w @ x_minus + b:+.6f}   (on the −1 plane)")
print(f"w·x⁺ + b = {w @ x_plus + b:+.6f}   (on the +1 plane, as claimed)")
print(f"M = ‖x⁺ − x⁻‖ = {np.linalg.norm(x_plus - x_minus):.6f}    2/‖w‖ = {2 / np.linalg.norm(w):.6f}")
# x⁺ is the CLOSEST point: any other point of the +1 plane is farther from x⁻
others = [point_on_plane(1.0) for _ in range(1000)]
print(f"closest of 1,000 random +1-plane points: {min(np.linalg.norm(o - x_minus) for o in others):.4f} ≥ M")

🔍 **What just happened?** Walking $\lambda = 2/w^\top w$ steps of $w$ from the −1 plane lands exactly on the +1 plane,
the distance travelled equals $2/\lVert w\rVert$, and no other point of the +1 plane is closer. The general fact
behind this — the distance from a point $x$ to the boundary is $|w^\top x + b|/\lVert w\rVert$ — is what
`geometric_margin` used in Section 1.

### 3.3 The optimisation problem (slides 17–21, 29, 31)

Maximising $M = 2/\lVert w\rVert$ is the same as minimising $\lVert w\rVert$, or — smoother and with a nicer
derivative — minimising $\tfrac12 w^\top w$. The margin lines must separate the classes: every +1 sample on or beyond
the +1 plane ($w^\top x_i + b \ge 1$), every −1 sample on or beyond the −1 plane ($w^\top x_i + b \le -1$). With $y_i \in
\{+1, -1\}$ both conditions are one:

$$\min_{w,\,b}\; \frac{1}{2} w^\top w \qquad \text{subject to}\qquad y_i\,(w^\top x_i + b) \ge 1, \quad i = 1, \dots, N .$$

*In words:* find the shortest weight vector (= widest margin) that puts every training sample on the correct side of
its margin line. There are N constraints, one per sample (slide 19: $n_1$ for one class, $n_2$ for the other).

🧮 **Refresher — quadratic programming (QP).** An optimisation problem whose objective is quadratic in the unknowns
and whose constraints are linear (in)equalities is a **quadratic program** (slide 20). Because $\tfrac12 w^\top w$ is a
bowl and the feasible region is an intersection of half-spaces, there is a single best solution and standard solvers
find it. We use SciPy's general-purpose `minimize` with the SLSQP method (sequential least-squares quadratic
programming), which accepts inequality constraints of the form $g(z) \ge 0$.

In [ ]:
# ── 3.4 Solve the hard-margin primal QP with SciPy ──
def hard_margin_primal(X, y):
    """min ½wᵀw  s.t.  yᵢ(wᵀxᵢ + b) ≥ 1 for all i.  Returns (w, b)."""
    J = X.shape[1]
    objective = lambda z: 0.5 * z[:J] @ z[:J]                   # z = (w_1, …, w_J, b)
    gradient = lambda z: np.r_[z[:J], 0.0]
    constraints = {"type": "ineq",                                # SLSQP wants g(z) ≥ 0
                   "fun": lambda z: y * (X @ z[:J] + z[J]) - 1,
                   "jac": lambda z: y[:, None] * np.c_[X, np.ones(len(y))]}
    res = minimize(objective, np.zeros(J + 1), jac=gradient, constraints=[constraints], method="SLSQP",
                   options={"ftol": 1e-12, "maxiter": 1000})
    assert res.success, res.message
    return res.x[:J], res.x[J]


w_hm, b_hm = hard_margin_primal(X_toy, y_toy)
scores = y_toy * (X_toy @ w_hm + b_hm)                           # yᵢ(wᵀxᵢ + b) for every sample
sv_hm = np.isclose(scores, 1.0, atol=1e-6)                       # constraints that are exactly active
print("w =", w_hm.round(4), " b =", round(b_hm, 4))
print(f"margin M = 2/‖w‖ = {2 / np.linalg.norm(w_hm):.3f}   (best of the random lines: {lines[-1][0]:.3f})")
print("smallest yᵢ(wᵀxᵢ + b) =", scores.min().round(6), "| support vectors (active constraints):", np.flatnonzero(sv_hm))

svc = SVC(kernel="linear", C=1e6).fit(X_toy, y_toy)                # a huge C ≈ hard margin
print("scikit-learn: w =", svc.coef_[0].round(4), " b =", svc.intercept_.round(4), " support vectors:", np.sort(svc.support_))

In [ ]:
# ── 3.5 Draw the maximum-margin solution ──
fig, ax = plt.subplots(figsize=(4.6, 4.4))
draw_margin(ax, w_hm, b_hm, X_toy, y_toy, sv=sv_hm,
            title=f"hard-margin SVM (SIMULATED toy)\nM = 2/‖w‖ = {2 / np.linalg.norm(w_hm):.2f}")
ax.legend(fontsize=7, loc="upper left"); plt.tight_layout(); plt.show()

🔍 **What just happened?** SciPy found the shortest $w$ that satisfies all 20 constraints. Its margin (4.25) beats
every one of the 1,411 random separating lines from Section 1 (the luckiest reached 4.22). Exactly three constraints are active — three training
points sit *on* the dashed margin lines with $y_i(w^\top x_i + b) = 1$: these are the support vectors, and
scikit-learn's `SVC` (with a huge C, which approximates the hard margin) reports the same three points and the same
$w, b$ to four decimals. In two dimensions three support vectors are typical: two on one side fix the
direction, one on the other side fixes the width.

### 3.6 The SVM ignores points far from the boundary — logistic regression does not

Slide 9's contrast, as an experiment: add 30 more type −1 tumours that are correctly classified and lie **outside**
the margin, and refit both models.

In [ ]:
# ── 3.6 Add 30 easy points: who moves? ──
rng = np.random.default_rng(SEED + 36)
X_extra = rng.normal([-3.0, -1.5], 0.4, (30, 2))
print("smallest yᵢ(wᵀxᵢ + b) of the new points under the old SVM:", (-(X_extra @ w_hm + b_hm)).min().round(2), "(> 1: outside the margin)")
X_more, y_more = np.vstack([X_toy, X_extra]), np.r_[y_toy, -np.ones(30)]
w_hm2, b_hm2 = hard_margin_primal(X_more, y_more)
logreg2 = LogisticRegression(C=1.0).fit(X_more, y_more)

angle = lambda vec: np.degrees(np.arctan2(vec[1], vec[0]))
offset = lambda vec, c: -c / np.linalg.norm(vec)                # signed distance of the boundary from the origin
print(f"SVM   : angle of w {angle(w_hm):6.2f}° → {angle(w_hm2):6.2f}°,  offset {offset(w_hm, b_hm):+.3f} → {offset(w_hm2, b_hm2):+.3f}")
print(f"LogReg: angle of w {angle(logreg.coef_[0]):6.2f}° → {angle(logreg2.coef_[0]):6.2f}°,  "
      f"offset {offset(logreg.coef_[0], logreg.intercept_[0]):+.3f} → {offset(logreg2.coef_[0], logreg2.intercept_[0]):+.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (w0, b0), (w1, b1), name in [(axes[0], (w_hm, b_hm), (w_hm2, b_hm2), "SVM"),
                                     (axes[1], (logreg.coef_[0], logreg.intercept_[0]),
                                      (logreg2.coef_[0], logreg2.intercept_[0]), "logistic regression")]:
    ax.scatter(*X_more[y_more > 0].T, s=18, color=CLASS_COLORS[1]); ax.scatter(*X_more[y_more < 0].T, s=18, color=CLASS_COLORS[0])
    ax.scatter(*X_extra.T, s=40, facecolor="none", edgecolor="k", lw=0.6, label="30 added tumours")
    x1 = np.linspace(-4.5, 4.5, 2)
    ax.plot(x1, (-b0 - w0[0] * x1) / w0[1], color="0.5", lw=2, label="before")
    ax.plot(x1, (-b1 - w1[0] * x1) / w1[1], "k--", lw=1.2, label="after")
    ax.set_xlim(-4.5, 4.5); ax.set_ylim(-4.5, 4.5); ax.set_aspect("equal"); ax.set_title(name)
    ax.set_xlabel("marker gene 1"); ax.set_ylabel("marker gene 2"); ax.legend(fontsize=7, loc="upper left")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The SVM boundary does not move at all: the new points are not support vectors, so they
do not enter the solution. The logistic-regression boundary tilts and shifts, because every point contributes to the
likelihood (and the extra 30 type −1 samples also change the class balance, which moves the intercept). Neither
behaviour is "right" in general: the SVM is robust to the bulk of the data but sensitive to the few points near the
boundary; logistic regression uses all the data and gives probabilities.

✏️ **Your turn.** Move the cloud of added points *into* the margin (e.g. centre `[-0.8, -0.8]`, spread 0.2) and re-run.
What happens to the SVM's support vectors and margin now — and can the hard-margin problem still be solved if one
added point lands on the wrong side?

### 📑 Slide connection — `007_SVM`, slides 15–31

- **What the slides show.** Learning as optimisation: what is the SVM's loss (15)? Maximising M means minimising its inverse, ½wᵀw, subject to N linear constraints (17–21, 29, 31), a quadratic program a standard solver can handle (20–21). The margin expression: w is perpendicular to the planes (Fact 1, slide 24); x⁺ = λw + x⁻ (Fact 2, 25); substitution gives λ = 2/wᵀw (27) and M = |x⁺ − x⁻| = λ‖w‖ = 2/√(wᵀw) (28).
- **What this notebook adds.** Each derivation step numerically verified in 5 dimensions (including that x⁺ is the closest point); the canonical-scaling argument for why the margin planes can be put at ±1; the hard-margin QP solved with SciPy's SLSQP and checked against scikit-learn; an experiment showing that points outside the margin do not move the SVM but do move logistic regression.
- **Exam tip.** The derivation of M = 2/‖w‖ from the four facts of slide 26 is a classic exam question — practise writing the five lines of slide 27 without looking. Remember why minimising ½wᵀw (not ‖w‖) is used: same minimiser, but a smooth quadratic objective.

---
## 4 · The soft margin: when no line separates the data

> 📍 **Where we are.** Real expression data are noisy; outliers and mislabelled samples make perfect separation
> impossible (slide 32), and then the hard-margin QP has **no feasible solution at all**. Slides 33–41 relax it.

Slide 33 suggests minimising the training error as well as maximising the margin. Counting misclassifications would
make the problem combinatorial, so slides 34–38 instead measure **how far each offending point is from its correct
margin plane** with a **slack variable** $\varepsilon_i \ge 0$:

$$\min_{w,\,b,\,\varepsilon}\; \frac12 w^\top w + C\sum_{i=1}^{N}\varepsilon_i \qquad\text{subject to}\qquad
  y_i\,(w^\top x_i + b) \ge 1 - \varepsilon_i, \quad \varepsilon_i \ge 0, \quad i = 1, \dots, N .$$

*In words:* every sample may violate its margin by $\varepsilon_i$, but each unit of violation costs $C$. Samples
beyond their margin line need no slack ($\varepsilon_i = 0$); a sample inside the band has $0 < \varepsilon_i \le 1$;
a misclassified sample has $\varepsilon_i > 1$. Now there are 2N constraints (slide 40), and the problem is always
feasible.

- **C is a knob** (slide 40: "like regularization in ridge regression"). Large C: violations are expensive → narrow
  margin, few violations, risk of fitting noise. Small C: violations are cheap → wide margin, many support vectors,
  smoother boundary. As $C \to \infty$ we recover the hard margin (when the data are separable).
- ⚠️ **Units.** The slides call $\varepsilon_i$ "the distance between these points and their correct plane". Strictly,
  $\varepsilon_i$ is measured in units of the score $w^\top x + b$; the geometric distance to the correct margin plane is
  $\varepsilon_i / \lVert w\rVert$.

In [ ]:
# ── 4.1 Overlapping data and the soft-margin primal QP ──
def soft_margin_primal(X, y, C: float):
    """min ½wᵀw + CΣεᵢ  s.t.  yᵢ(wᵀxᵢ + b) ≥ 1 − εᵢ, εᵢ ≥ 0.  Returns (w, b, ε)."""
    N, J = X.shape
    # z = (w, b, ε_1 … ε_N). For large C we divide the objective by C: same minimiser, better-scaled numbers
    s = max(C, 1.0)
    objective = lambda z: (0.5 * z[:J] @ z[:J] + C * z[J + 1:].sum()) / s
    gradient = lambda z: np.r_[z[:J], 0.0, C * np.ones(N)] / s
    A = np.c_[y[:, None] * X, y, np.eye(N)]                               # rows: yᵢxᵢ, yᵢ, e_i
    constraints = {"type": "ineq", "fun": lambda z: A @ z - 1, "jac": lambda z: A}
    bounds = [(None, None)] * (J + 1) + [(0, None)] * N
    res = minimize(objective, np.r_[np.zeros(J + 1), np.ones(N)], jac=gradient, bounds=bounds,
                   constraints=[constraints], method="SLSQP", options={"ftol": 1e-12, "maxiter": 2000})
    assert res.success, res.message
    return res.x[:J], res.x[J], res.x[J + 1:]


rng = np.random.default_rng(SEED + 41)
X_ov, y_ov = two_tumour_types(rng, n=40, spread=1.0, shift=1.0)          # SIMULATED, overlapping clouds
rows, fits = [], {}
for C in [0.01, 0.1, 1.0, 100.0]:
    w_c, b_c, eps = soft_margin_primal(X_ov, y_ov, C)
    fits[C] = (w_c, b_c, eps)
    t = y_ov * (X_ov @ w_c + b_c)
    rows.append(dict(C=C, margin_M=2 / np.linalg.norm(w_c), support_vectors=int(np.sum(t <= 1 + 1e-5)),
                     inside_margin=int(np.sum((t > 0) & (t < 1 - 1e-5))), misclassified=int(np.sum(t < 0)),
                     total_slack=eps.sum()))
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
# ── 4.2 The effect of C (slide 73) ──
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, C in zip(axes, [0.01, 1.0, 100.0]):
    w_c, b_c, eps = fits[C]
    sv = y_ov * (X_ov @ w_c + b_c) <= 1 + 1e-5
    draw_margin(ax, w_c, b_c, X_ov, y_ov, sv=sv, title=f"C = {C:g}: M = {2 / np.linalg.norm(w_c):.2f}, {sv.sum()} SVs")
axes[0].legend(fontsize=7, loc="upper left")
plt.suptitle("soft-margin SVM on overlapping SIMULATED tumours", fontsize=10); plt.tight_layout(); plt.show()

🔍 **What just happened?** With $C = 0.01$ violations are almost free: the margin is very wide ($M = 5.14$) and 36
of the 40 tumours are support vectors (inside the band or on its edge). As C grows the band narrows ($M$ = 2.58,
1.22, 1.05) and the number of support vectors falls (18, 12, 10); the total slack shrinks from 19.5 to 8.2. Four
tumours stay misclassified for every C ≥ 0.1 (five at C = 0.01): they lie deep in the other cloud and no line can fix
them — paying more for violations only squeezes the margin. This is the trade-off of slide 73 (ESL Fig. 12.2, C = 10,000 vs
C = 0.01): the slide's speaker note says it in one line — *large C emphasises training-error minimisation, the margin
gets smaller.*

### 4.3 The soft margin is a loss function: the hinge (slide 70)

At the optimum each slack is as small as its constraint allows: $\varepsilon_i = \max(0,\, 1 - y_i(w^\top x_i + b))$.
Writing $t_i = y_i(w^\top x_i + b)$ (positive when sample $i$ is classified correctly) and $[u]_+ = \max(0, u)$,
the constrained problem becomes an unconstrained one:

$$\min_{w,\,b}\; \frac12 w^\top w + C\sum_{i=1}^{N} \big[\,1 - t_i\,\big]_+ .$$

*In words:* a ridge penalty on $w$ (Module 07) plus the **hinge loss** of every sample. A sample with $t_i \ge 1$
(correct, beyond the margin) costs nothing — that is why points far from the boundary do not matter (Section 3.6). The
ideal 0–1 loss (1 if $t < 0$, else 0) is not differentiable and not convex; the hinge is a convex upper bound on it.
Logistic regression's loss is $\log(1 + e^{-t})$ — a smooth cousin of the hinge that never quite reaches zero, so
*every* point keeps a little influence.

In [ ]:
# ── 4.3 Three loss functions of the margin score t = y(wᵀx + b) ──
t = np.linspace(-2, 3, 501)
fig, ax = plt.subplots(figsize=(5.4, 3.4))
ax.plot(t, (t < 0).astype(float), color="tab:green", lw=2, label="0–1 loss (ideal classifier)")
ax.plot(t, np.maximum(0, 1 - t), color="tab:red", lw=2, label="hinge loss $[1-t]_+$ (SVM)")
ax.plot(t, np.log2(1 + np.exp(-t)), color="tab:blue", lw=2, ls="--", label=r"logistic loss $\log_2(1+e^{-t})$")
ax.axvline(0, color="k", lw=0.5); ax.axvline(1, color="k", lw=0.5, ls=":")
ax.set_xlabel("t = y (wᵀx + b)   (> 0 means correctly classified)"); ax.set_ylabel("loss")
ax.set_title("SVM and logistic regression both bound the 0–1 loss"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

The logistic loss is drawn in base 2 so that it equals 1 at $t = 0$ like the others; in base $e$ it is the negative
log-likelihood of Module 09. The unconstrained form also gives us a second, very simple way to train an SVM.

🧮 **Refresher — subgradients.** The hinge has a kink at $t = 1$, where the derivative is undefined. A
**subgradient** is any slope between the left and right derivatives; for $[1 - t]_+$ we can use $-1$ when $t < 1$ and
$0$ otherwise. Gradient descent with subgradients and slowly shrinking step sizes $\eta_k = \eta_0/\sqrt{k}$ still
converges for convex functions (Module 04 introduced gradient descent). The subgradient of the objective is

$$g_w = w - C\sum_{i:\,t_i<1} y_i x_i, \qquad g_b = -C\sum_{i:\,t_i<1} y_i .$$

*In words:* the ridge term pulls $w$ towards 0; every sample that violates its margin pulls $w$ towards classifying it
better; samples with $t_i \ge 1$ exert no pull.

In [ ]:
# ── 4.4 Train the soft-margin primal by subgradient descent, from scratch ──
def svm_objective(w, b, X, y, C: float) -> float:
    """½wᵀw + C Σ hinge."""
    return 0.5 * w @ w + C * np.maximum(0, 1 - y * (X @ w + b)).sum()


def svm_subgradient_descent(X, y, C: float, n_iter: int = 3000, eta0: float = 0.05):
    """Subgradient descent on the hinge-loss objective; returns the best (w, b) seen and the objective history."""
    w, b = np.zeros(X.shape[1]), 0.0
    best, history = (np.inf, w, b), []
    for k in range(1, n_iter + 1):
        violators = y * (X @ w + b) < 1
        g_w = w - C * (y[violators, None] * X[violators]).sum(axis=0)
        g_b = -C * y[violators].sum()
        eta = eta0 / np.sqrt(k)                                  # shrinking steps
        w, b = w - eta * g_w, b - eta * g_b
        history.append(svm_objective(w, b, X, y, C))
        if history[-1] < best[0]:                               # subgradient steps are not monotone: keep the best
            best = (history[-1], w.copy(), b)
    return best[1], best[2], np.array(history)


w_qp, b_qp, _ = fits[1.0]
t0 = time.time()
w_sg, b_sg, hist = svm_subgradient_descent(X_ov, y_ov, C=1.0)
print(f"subgradient descent ({time.time() - t0:.2f} s): objective {svm_objective(w_sg, b_sg, X_ov, y_ov, 1.0):.4f}"
      f"  w = {w_sg.round(3)}  b = {b_sg:.3f}")
print(f"QP solution (4.1)          : objective {svm_objective(w_qp, b_qp, X_ov, y_ov, 1.0):.4f}"
      f"  w = {w_qp.round(3)}  b = {b_qp:.3f}")
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(np.arange(1, len(hist) + 1), hist - svm_objective(w_qp, b_qp, X_ov, y_ov, 1.0), color="tab:purple")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("iteration k"); ax.set_ylabel("objective − QP optimum")
ax.set_title("subgradient descent creeps towards the QP optimum"); plt.tight_layout(); plt.show()

🔍 **What just happened?** Plain subgradient descent on the hinge objective reaches the QP's objective value to about
four significant digits and nearly the same $w$ and $b$. It is slow and noisy (note the log axes and the
non-monotone curve), but it needs nothing except a loop — the same algorithm, run on random mini-batches, is how
linear SVMs are trained on millions of samples (e.g. the Pegasos algorithm), and it is exactly how neural networks are
trained (Module 22).

✏️ **Your turn.** Re-run 4.1–4.2 with `spread=1.4` (more overlap). How do the number of support vectors and the
margin at C = 1 change? Then try `eta0=0.5` in 4.4: what happens to the objective curve?

### 📑 Slide connection — `007_SVM`, slides 32–41, 69–73

- **What the slides show.** Non-separable data (noise, outliers, 32). Instead of counting misclassified points, minimise their distance to the correct plane with slacks εᵢ: minimise wᵀw/2 + CΣεᵢ subject to wᵀxᵢ + b ≥ 1 − εᵢ (class +1), ≤ −1 + εᵢ (class −1) and εᵢ ≥ 0 — 2n constraints, 'like regularization in ridge regression' (33–41, 69, 72). The hinge loss [1 − t]₊ with t = (w'xᵢ + b)yᵢ versus the ideal 0–1 loss (70). Large C gives a smaller margin (73, ESL Fig. 12.2).
- **What this notebook adds.** The soft-margin primal solved with SciPy for four values of C, with margin width, support vectors, violators and total slack tabulated; the derivation that the constrained problem equals ridge + hinge; a plot of 0–1, hinge and logistic losses; a from-scratch subgradient-descent trainer checked against the QP; the note that εᵢ is a distance only after dividing by ‖w‖.
- **Exam tip.** Be able to say which samples have εᵢ = 0, 0 < εᵢ ≤ 1 and εᵢ > 1, and what happens to the margin and the number of support vectors as C increases.

---
## 5 · Duality: where the support vectors come from

> 📍 **Where we are.** We can solve the primal problem. Slide 42 announces a reformulation, the **dual**, for two
> reasons: it identifies the support vectors explicitly, and it enables the kernel trick of Section 6. The tool is the
> Lagrange multiplier (Module 04).

### 5.1 A one-dimensional warm-up (slide 46)

**Primal problem:** $\min_x x^2$ subject to $x \ge b$. Attach a **Lagrange multiplier** $\alpha \ge 0$ to the
constraint and form the **Lagrangian**

$$L(x, \alpha) = x^2 - \alpha\,(x - b), \qquad \alpha \ge 0 .$$

*In words:* the constraint $x - b \ge 0$ enters with a "price" $\alpha$. For a feasible $x$ the term $-\alpha(x-b)$
is $\le 0$, so maximising over $\alpha \ge 0$ gives back $x^2$; for an infeasible $x$ it can be made arbitrarily
large. Hence $\min_x \max_{\alpha\ge0} L$ is the original problem. The **dual** swaps the order: first minimise over
$x$ for fixed $\alpha$ — set the derivative to zero, $\frac{d}{dx}[x^2 - \alpha(x-b)] = 2x - \alpha = 0$, so
$x = \alpha/2$ — and substitute:

$$g(\alpha) = \Big(\frac{\alpha}{2}\Big)^2 - \alpha\Big(\frac{\alpha}{2} - b\Big) = -\frac{\alpha^2}{4} + \alpha b,
  \qquad \text{dual problem: } \max_{\alpha \ge 0}\; g(\alpha).$$

*In words:* the dual function $g(\alpha)$ is the best the Lagrangian can do at price $\alpha$; it is always a **lower
bound** on the primal optimum (weak duality), and for convex problems like this one the best lower bound equals the
optimum (strong duality).

In [ ]:
# ── 5.1 Primal and dual of min x² s.t. x ≥ b, for b = 1 and b = −1 ──
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
xs, alphas = np.linspace(-2, 3, 400), np.linspace(0, 4, 400)
for b_val, col in [(1.0, "tab:red"), (-1.0, "tab:blue")]:
    x_star = max(b_val, 0.0)                        # primal solution: the bowl's bottom if feasible, else the edge
    g = -alphas ** 2 / 4 + alphas * b_val
    a_star = max(2 * b_val, 0.0)                    # dual solution: g'(α) = −α/2 + b = 0, clipped at α ≥ 0
    d_star = -a_star ** 2 / 4 + a_star * b_val + 0.0
    print(f"b = {b_val:+.0f}: primal x* = {x_star:.2f}, p* = {x_star ** 2:.3f} | dual α* = {a_star:.2f}, "
          f"d* = {d_star:.3f} | α*(x* − b) = {a_star * (x_star - b_val):.3f}")
    axes[0].plot(xs, xs ** 2, color="0.6"); axes[0].axvspan(b_val, 3, color=col, alpha=0.08)
    axes[0].plot(x_star, x_star ** 2, "o", color=col, label=f"b = {b_val:+.0f}: feasible x ≥ b (shaded), optimum")
    axes[1].plot(alphas, g, color=col, label=f"b = {b_val:+.0f}: g(α) = −α²/4 + αb")
    axes[1].plot(a_star, d_star, "o", color=col)
axes[0].set_xlabel("primal variable x"); axes[0].set_ylabel("x²"); axes[0].set_title("primal: min x² s.t. x ≥ b")
axes[1].set_xlabel("dual variable α ≥ 0"); axes[1].set_ylabel("g(α)"); axes[1].set_title("dual: max g(α)")
axes[1].set_ylim(-4, 1.5)
for ax in axes:
    ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** For $b = 1$ the constraint binds: the primal optimum is $x^* = 1$ with value 1, and the
dual optimum is $\alpha^* = 2$ with the same value 1 (strong duality). For $b = -1$ the constraint is irrelevant (the
bowl's bottom $x = 0$ is feasible): $x^* = 0$ and $\alpha^* = 0$. In both cases $\alpha^*(x^* - b) = 0$ — **complementary
slackness**: *a multiplier can be non-zero only if its constraint is active.* This single fact explains support
vectors: in the SVM, every training sample has a constraint and a multiplier $\alpha_i$, and only samples whose
constraint is active (on the margin) or violated can have $\alpha_i > 0$.

### 5.2 The SVM dual, step by step (slides 47–51)

The primal is $\min_{w,b} \tfrac12 w^\top w$ subject to $y_i(w^\top x_i + b) - 1 \ge 0$ for all $i$. One multiplier
$\alpha_i \ge 0$ per sample gives the Lagrangian

$$L(w, b, \alpha) = \frac12 w^\top w - \sum_{i=1}^{N} \alpha_i\big[\,y_i(w^\top x_i + b) - 1\,\big], \qquad \alpha_i \ge 0 .$$

*In words:* the objective minus a price $\alpha_i$ times each sample's "surplus" $y_i(w^\top x_i + b) - 1$ — one price
per training sample, exactly as in the warm-up.

⚠️ **A sign typo on slides 48 and 51.** The Lagrangian box there says "$\alpha_i \le 0\ \forall i$". With the
constraint written as $y_i(w^\top x_i + b) - 1 \ge 0$ and subtracted, the multipliers must be **non-negative**,
$\alpha_i \ge 0$ — as the dual box on the same slides correctly states.

🧮 **Refresher — derivatives with respect to a vector** (Module 04). $\partial(\tfrac12 w^\top w)/\partial w = w$ and
$\partial(w^\top x)/\partial w = x$ — the same rules as for numbers, applied entry by entry. Setting the derivatives of
$L$ to zero (slide 48, "differentiate w.r.t. w, b"):

$$\frac{\partial L}{\partial w} = w - \sum_i \alpha_i y_i x_i = 0 \;\Rightarrow\; \boxed{w = \sum_{i=1}^{N}\alpha_i y_i x_i},
\qquad \frac{\partial L}{\partial b} = -\sum_i \alpha_i y_i = 0 \;\Rightarrow\; \sum_{i=1}^{N}\alpha_i y_i = 0 .$$

*In words:* the optimal weight vector is a weighted sum of training samples — and only samples with $\alpha_i \ne 0$
contribute. Substituting $w$ back into $L$ (the $b$ term vanishes because $\sum_i\alpha_i y_i = 0$, and
$\tfrac12 w^\top w - \sum_i \alpha_i y_i w^\top x_i = -\tfrac12 w^\top w$):

$$\max_{\alpha}\;\; \sum_{i=1}^{N}\alpha_i - \frac12\sum_{i=1}^{N}\sum_{j=1}^{N}\alpha_i\alpha_j\, y_i y_j\, x_i^\top x_j
\qquad\text{subject to}\qquad \sum_{i=1}^{N}\alpha_i y_i = 0, \quad \alpha_i \ge 0 .$$

*In words:* the dual has one unknown per **sample** (not per feature), and the data enter **only through dot
products** $x_i^\top x_j$ — the door to kernels. For the soft margin (slide 52 and the extra slide) the derivation
gains a multiplier for each $\varepsilon_i \ge 0$, and the only change in the dual is a **box constraint**
$0 \le \alpha_i \le C$: C caps how hard any single (possibly noisy) sample can pull on the boundary.

In [ ]:
# ── 5.3 Solve the dual QP with SciPy (any kernel matrix K; C=None means hard margin) ──
def svm_dual(K, y, C=None):
    """max Σα − ½ αᵀQα with Q_ij = yᵢyⱼK_ij, s.t. Σαᵢyᵢ = 0 and 0 ≤ αᵢ ≤ C. Returns α."""
    N = len(y)
    Q = np.outer(y, y) * K
    objective = lambda a: 0.5 * a @ Q @ a - a.sum()            # minimise the negative of the dual
    gradient = lambda a: Q @ a - 1.0
    equality = {"type": "eq", "fun": lambda a: a @ y, "jac": lambda a: y}
    res = minimize(objective, np.zeros(N), jac=gradient, bounds=[(0.0, C)] * N, constraints=[equality],
                   method="SLSQP", options={"ftol": 1e-12, "maxiter": 2000})
    assert res.success, res.message
    return res.x


def intercept_from_alpha(K, y, alpha, C=None, tol: float = 1e-6) -> float:
    """b from the margin support vectors (0 < αᵢ < C), where yᵢ(Σⱼ αⱼyⱼK_ji + b) = 1 holds exactly."""
    upper = np.inf if C is None else C
    on_margin = (alpha > tol) & (alpha < upper - tol)
    return float(np.mean(y[on_margin] - (alpha * y) @ K[:, on_margin]))


alpha_hm = svm_dual(X_toy @ X_toy.T, y_toy)                   # linear kernel: K_ij = xᵢ·xⱼ
w_dual = (alpha_hm * y_toy) @ X_toy                           # w = Σ αᵢ yᵢ xᵢ
b_dual = intercept_from_alpha(X_toy @ X_toy.T, y_toy, alpha_hm)
print("non-zero α:", {int(i): round(float(a), 4) for i, a in enumerate(alpha_hm) if a > 1e-6})
print("w from the dual =", w_dual.round(4), " b =", round(b_dual, 4))
print("w from the primal =", w_hm.round(4), " b =", round(b_hm, 4))
print("Σ αᵢ yᵢ =", round(alpha_hm @ y_toy, 10), "| support vectors from the primal (active constraints):", np.flatnonzero(sv_hm))

🔍 **What just happened?** The dual returns 20 multipliers, of which only three are non-zero — for exactly the three
samples whose primal constraints were active in Section 3.4. Building $w = \sum_i \alpha_i y_i x_i$ from those three
samples reproduces the primal $w$ and $b$. This is slide 49's statement: **$w$ is defined only by the $\alpha_i$ that
are not 0, and those data points are the support vectors.**

### 5.4 Soft-margin support vectors: three kinds of samples (slides 52, 56 and the extra slide)

With the box $0 \le \alpha_i \le C$, complementary slackness sorts the samples into three groups by their score
$t_i = y_i(w^\top x_i + b)$:

| multiplier | position | score $t_i$ | slack $\varepsilon_i$ |
|---|---|---|---|
| $\alpha_i = 0$ | beyond its margin line (not a support vector) | $\ge 1$ | 0 |
| $0 < \alpha_i < C$ | exactly **on** its margin line | $= 1$ | 0 |
| $\alpha_i = C$ | inside the band or on the wrong side | $\le 1$ | $1 - t_i$ |

Slide 56 describes support vectors in the non-separable case as "data points on the wrong side of the margin"; the
complete statement includes the margin-line points with $0 < \alpha_i < C$. Let us solve the dual on the overlapping
data and compare with scikit-learn, whose `SVC` stores the support-vector indices in `support_` and the products
$\alpha_i y_i$ in `dual_coef_`.

In [ ]:
# ── 5.4 Soft-margin dual: categories of α, complementary slackness, check against scikit-learn ──
C_demo = 1.0
K_ov = X_ov @ X_ov.T
alpha = svm_dual(K_ov, y_ov, C=C_demo)
w_d = (alpha * y_ov) @ X_ov
b_d = intercept_from_alpha(K_ov, y_ov, alpha, C=C_demo)
t_d = y_ov * (X_ov @ w_d + b_d)
tol = 1e-5
groups = {"α = 0": alpha <= tol, "0 < α < C": (alpha > tol) & (alpha < C_demo - tol), "α = C": alpha >= C_demo - tol}
for name, m in groups.items():
    print(f"{name:10s}: {m.sum():2d} samples, scores t from {t_d[m].min():+.3f} to {t_d[m].max():+.3f}")
eps_d = np.maximum(0, 1 - t_d)
print("complementary slackness  max |αᵢ (tᵢ − 1 + εᵢ)| =", np.abs(alpha * (t_d - 1 + eps_d)).max().round(6))

svc = SVC(kernel="linear", C=C_demo).fit(X_ov, y_ov)
ours = np.flatnonzero(alpha > tol)
print("\nsupport vectors — ours:", ours)
print("support vectors — sklearn:", np.sort(svc.support_), "| same set:", set(ours) == set(svc.support_))
print("max |αᵢyᵢ (ours) − dual_coef_ (sklearn)| =",
      np.abs((alpha * y_ov)[svc.support_] - svc.dual_coef_[0]).max().round(4))
print("w ours =", w_d.round(4), " sklearn =", svc.coef_[0].round(4), "| b ours =", round(b_d, 4), " sklearn =", svc.intercept_.round(4))

🔍 **What just happened?** The three groups behave exactly as the table predicts: samples with $\alpha_i = 0$ have
scores of at least 1, the few with $0 < \alpha_i < C$ sit at $t_i = 1$ on their margin line, and those at the cap
$\alpha_i = C$ have $t_i < 1$ — including the misclassified ones with $t_i < 0$. Complementary slackness holds to solver
precision, and scikit-learn (which solves the same dual with the SMO algorithm of LIBSVM) finds the same support
vectors, the same $\alpha_i y_i$ and the same $w$ and $b$ to at least four decimals.

⚠️ **Another small slide slip.** Slide 52 and the extra slide write the box as "$C > \alpha_i \ge 0$". The bound is
**$C \ge \alpha_i \ge 0$**: the value $\alpha_i = C$ is attained — by every margin violator, as the output shows.

✏️ **Your turn.** Re-run 5.4 with `C_demo = 0.1` and with `C_demo = 100`. How do the sizes of the three groups change?
Then solve the soft-margin dual on the *separable* toy (`svm_dual(X_toy @ X_toy.T, y_toy, C=1.0)`): does any sample
reach α = C?

### 5.5 Two parameterisations, two prediction costs (slides 54–55)

| | primal | dual |
|---|---|---|
| parameters | $w$, $b$ | $\alpha_1, \dots, \alpha_N$ (and $b$) |
| how many | one per **feature** (J) | one per **training sample** (N) |
| meaning | direction and position of the boundary | importance of each sample (support vectors) |
| prediction for a new $x$ | $\operatorname{sign}(w^\top x + b)$: J multiplications | $\operatorname{sign}\big(\sum_{i\in SV}\alpha_i y_i\, x_i^\top x + b\big)$: J × r multiplications |

$r$ is the number of support vectors (slide 55 writes $m$ for the number of features). With a linear kernel the primal
is cheaper at test time; the dual's advantage is that it needs only dot products — which, in the next section, we will
replace by something much more powerful.

In [ ]:
# ── 5.5 The dual prediction uses only the support vectors ──
x_new = np.array([0.5, -0.2])
sv_idx = np.flatnonzero(alpha > tol)
f_primal = w_d @ x_new + b_d
f_dual = np.sum(alpha[sv_idx] * y_ov[sv_idx] * (X_ov[sv_idx] @ x_new)) + b_d
print(f"f(x_new): primal {f_primal:+.6f}  dual {f_dual:+.6f}")
print(f"cost: primal J = {X_ov.shape[1]} multiplications; dual J × r = {X_ov.shape[1]} × {len(sv_idx)} = {X_ov.shape[1] * len(sv_idx)}")

### 📑 Slide connection — `007_SVM`, slides 42–56 and 007_SVM_extra_slides

- **What the slides show.** Switch to the dual to identify support vectors and enable the kernel trick (42). The toy primal–dual pair min x² s.t. x ≥ b ↔ max −α²/4 + αb s.t. α ≥ 0, via the Lagrangian min_x max_α x² − α(x − b) (46). The SVM Lagrangian; differentiating gives w = Σαᵢxᵢyᵢ and the dual max Σαᵢ − ½ΣΣαᵢαⱼyᵢyⱼxᵢ'xⱼ s.t. Σαᵢyᵢ = 0, αᵢ ≥ 0 (47–51); non-zero αᵢ define the support vectors and prediction needs dot products with them (49–50); in the non-separable case the αᵢ are bounded by C, which keeps α from going to ∞ and prevents overfitting to noisy samples (52, extra slide); parameter counts (54); test cost m vs m·r (55); support vectors in the non-separable case (56).
- **What this notebook adds.** The 1-D primal and dual solved and plotted for b = ±1 with strong duality and complementary slackness; the SVM dual derived with every substitution shown; both QPs solved with SciPy and shown to give the same w and b; the three α categories verified numerically; a check of support_, dual_coef_, coef_ and intercept_ against scikit-learn. Two slide slips corrected: the Lagrangian box on slides 48/51 says αᵢ ≤ 0 (should be ≥ 0), and slides 52/extra write C > αᵢ (should be C ≥ αᵢ).
- **Exam tip.** Be able to derive w = Σαᵢyᵢxᵢ and Σαᵢyᵢ = 0 from the Lagrangian, and to classify a sample as non-SV, margin SV or violator from its αᵢ (0, between 0 and C, equal to C).

---
## 6 · Kernels: curved boundaries from dot products

> 📍 **Where we are.** Everything so far draws straight lines. Biology is rarely that kind: a cell can be abnormal if
> a marker is either too low *or* too high. Slides 58–67 show how the dual makes curved boundaries cheap.

### 6.1 One dimension: separable after a transformation (slides 58–60)

Slide 58 shows five samples on a line, ordered red, red, blue, blue, red: no single threshold separates them. Slide 60
maps each $x$ to a point in two dimensions, $\phi(x) = (x, x^2)$, where a straight line *does* separate the classes.

In [ ]:
# ── 6.1 Five points on a line; a linear SVM in the feature space φ(x) = (x, x²) ──
x1d = np.array([1.0, 2.0, 3.0, 4.0, 6.0])
y1d = np.array([+1, +1, -1, -1, +1], dtype=float)              # red, red, blue, blue, red (as on slide 58)
Phi = np.c_[x1d, x1d ** 2]
w_phi, b_phi = hard_margin_primal(Phi, y1d)
roots = np.sort(np.roots([w_phi[1], w_phi[0], b_phi]).real)     # where w₁x + w₂x² + b = 0 back on the line
print("w =", w_phi.round(4), " b =", round(b_phi, 4), "→ boundary points on the original axis:", roots.round(3))
print("training predictions:", np.sign(Phi @ w_phi + b_phi), " true:", y1d)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
cols = [CLASS_COLORS[1] if v > 0 else CLASS_COLORS[0] for v in y1d]
axes[0].scatter(x1d, np.zeros(5), c=cols, s=60, zorder=3)
for r in roots:
    axes[0].axvline(r, color="k", ls="--", lw=1)
axes[0].set_yticks([]); axes[0].set_xlabel("x (one gene)"); axes[0].set_title("1-D input: no single threshold works\n(dashed: the SVM's two thresholds)")
grid = np.linspace(0, 7, 200)
axes[1].plot(grid, grid ** 2, color="0.85", lw=1, label="all points (x, x²)")
axes[1].scatter(Phi[:, 0], Phi[:, 1], c=cols, s=60, zorder=3)
axes[1].plot(grid, (-b_phi - w_phi[0] * grid) / w_phi[1], "k-", lw=1.2, label="linear SVM boundary in φ-space")
axes[1].set_ylim(-3, 40); axes[1].set_xlabel("x"); axes[1].set_ylabel("x²"); axes[1].legend(fontsize=7)
axes[1].set_title("φ(x) = (x, x²): one straight line separates")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In the $(x, x^2)$ plane the points lie on a parabola, and a straight line cuts off the two
blue points. Back on the original axis that single line becomes **two thresholds** — "blue if $x$ is between about 2.4
and 5.6" (the roots of $w_2x^2 + w_1x + b = 0$). A linear classifier in a richer feature space is a non-linear classifier in the original space (slide 64:
$w^\top x + b \to v^\top\phi(x) + d$).

### 6.2 The kernel trick (slides 61–66)

For two-dimensional inputs, slide 62 uses $\phi(x) = (x_1^2,\; x_2^2,\; \sqrt2\,x_1x_2)$. Two problems loom (slide 64):
high-dimensional $\phi$ is expensive to compute, and the primal has one parameter per feature. The dual solves both,
because it never needs $\phi(x)$ itself — only dot products $\phi(x_i)^\top\phi(x_j)$. And those can be computed
without forming $\phi$:

$$\phi(x)^\top\phi(z) = x_1^2 z_1^2 + x_2^2 z_2^2 + 2\,x_1x_2\,z_1z_2 = (x_1z_1 + x_2z_2)^2 = (x^\top z)^2 \equiv \kappa(x, z).$$

*In words:* square the ordinary dot product and you get the dot product in the 3-D feature space for free. A function
$\kappa(x, z)$ that equals $\phi(x)^\top\phi(z)$ for some feature map is a **kernel**. Slides 65–66 do the same with
$\phi(x) = (1, \sqrt2x_1, \sqrt2x_2, x_1^2, x_2^2, \sqrt2x_1x_2)$ and the polynomial kernel
$\kappa(x, z) = (\gamma\,x^\top z + r)^M$ with $M = 2$, $\gamma = r = 1$. Check both identities:

In [ ]:
# ── 6.2 Verify the kernel identities numerically ──
def phi3(v):
    """Slide 62's feature map for 2-D inputs."""
    return np.array([v[0] ** 2, v[1] ** 2, np.sqrt(2) * v[0] * v[1]])


def phi6(v):
    """Slides 65–66's feature map for 2-D inputs."""
    return np.array([1.0, np.sqrt(2) * v[0], np.sqrt(2) * v[1], v[0] ** 2, v[1] ** 2, np.sqrt(2) * v[0] * v[1]])


rng = np.random.default_rng(SEED + 62)
for _ in range(3):
    x, z = rng.normal(size=2), rng.normal(size=2)
    print(f"φ3(x)·φ3(z) = {phi3(x) @ phi3(z):+.10f}   (xᵀz)²   = {(x @ z) ** 2:+.10f}   |   "
          f"φ6(x)·φ6(z) = {phi6(x) @ phi6(z):+.10f}   (xᵀz+1)² = {(x @ z + 1) ** 2:+.10f}")

Slide 65 asks about the **time complexity**. For J-dimensional inputs (genes!), the explicit map behind
$(x^\top z + 1)^2$ has $1 + J + J + J(J-1)/2 = (J+1)(J+2)/2$ entries; the kernel needs one J-term dot product. Let us
time both for growing J.

In [ ]:
# ── 6.3 Explicit features vs kernel: cost as the number of genes J grows ──
def phi_quadratic(v):
    """Explicit feature map of (vᵀu + 1)² for J-dimensional v: [1, √2 v, v², √2 vᵢvⱼ (i < j)]."""
    iu = np.triu_indices(len(v), 1)
    return np.concatenate([[1.0], np.sqrt(2) * v, v ** 2, np.sqrt(2) * np.outer(v, v)[iu]])


rng = np.random.default_rng(SEED + 63)
for J in [10, 100, 1000, 2000]:
    x, z = rng.normal(size=J), rng.normal(size=J)
    t0 = time.perf_counter(); explicit = phi_quadratic(x) @ phi_quadratic(z); t1 = time.perf_counter()
    kern = (x @ z + 1) ** 2; t2 = time.perf_counter()
    print(f"J = {J:5d}: φ has {(J + 1) * (J + 2) // 2:>9,d} entries | explicit {explicit:12.4f} in {1e3 * (t1 - t0):7.3f} ms"
          f" | kernel {kern:12.4f} in {1e3 * (t2 - t1):6.3f} ms")

🔍 **What just happened?** Both routes give the same number, but the explicit features grow quadratically: for 2,000
genes, two million features per sample and a time that grows with them, against a few thousand multiplications for the
kernel. For the 3,571 Golub genes the explicit quadratic map would have 6.4 million entries per patient. For the
Gaussian (RBF) kernel below the feature space is infinite-dimensional, and the kernel is the *only* way to compute it.

### 6.4 A kernel SVM from our own dual solver

Replace every $x_i^\top x_j$ in the dual by $\kappa(x_i, x_j)$ and every prediction $w^\top x$ by
$\sum_i \alpha_i y_i\,\kappa(x_i, x)$ (slide 63) — nothing else changes. We try it on a classic shape no line can
separate: two concentric rings, like an inner population of cells surrounded by another in a two-marker plot.

In [ ]:
# ── 6.4 Concentric rings: a polynomial-kernel SVM from our dual solver vs scikit-learn ──
X_ring, y_ring = make_circles(n_samples=80, noise=0.1, factor=0.4, random_state=SEED)
y_ring = np.where(y_ring == 1, 1.0, -1.0)                       # inner ring = +1
poly2 = lambda A, B: (A @ B.T) ** 2                             # κ(x, z) = (xᵀz)², slide 62

t0 = time.time()
C_ring = 10.0
alpha_r = svm_dual(poly2(X_ring, X_ring), y_ring, C=C_ring)
b_r = intercept_from_alpha(poly2(X_ring, X_ring), y_ring, alpha_r, C=C_ring)
decision_r = lambda Z: (alpha_r * y_ring) @ poly2(X_ring, Z) + b_r
sv_r = np.flatnonzero(alpha_r > 1e-6)
print(f"our dual: {len(sv_r)} support vectors, training accuracy {accuracy(y_ring, np.sign(decision_r(X_ring))):.3f}"
      f" ({time.time() - t0:.1f} s)")
svc_r = SVC(kernel="poly", degree=2, gamma=1.0, coef0=0.0, C=C_ring).fit(X_ring, y_ring)   # (1·xᵀz + 0)²
print("scikit-learn: support vectors", np.sort(svc_r.support_), "| ours", sv_r, "| b", svc_r.intercept_.round(3), "vs", round(b_r, 3))

linear_ring = SVC(kernel="linear", C=C_ring).fit(X_ring, y_ring)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_decision_regions(lambda Z: (linear_ring.predict(Z) > 0).astype(int), X_ring, (y_ring > 0).astype(int), ax=axes[0],
                      pad=0.3, title=f"linear SVM: training accuracy {linear_ring.score(X_ring, y_ring):.2f}")
plot_decision_regions(lambda Z: (decision_r(Z) > 0).astype(int), X_ring, (y_ring > 0).astype(int), ax=axes[1],
                      pad=0.3, title=f"kernel (xᵀz)² SVM, our dual: {len(sv_r)} support vectors")
axes[1].scatter(*X_ring[sv_r].T, s=120, facecolor="none", edgecolor="k")
for ax in axes:
    ax.set_xlabel("marker 1"); ax.set_ylabel("marker 2"); ax.set_aspect("equal")
plt.suptitle("SIMULATED rings", fontsize=10); plt.tight_layout(); plt.show()

🔍 **What just happened?** A linear SVM is helpless on rings (it can do little better than predicting one class). The
same dual solver with $\kappa(x, z) = (x^\top z)^2$ draws an ellipse — a straight plane in slide 62's 3-D space
$(x_1^2, x_2^2, \sqrt2x_1x_2)$ — and classifies the training data correctly using only a handful of support vectors.
scikit-learn's polynomial kernel with `degree=2, gamma=1, coef0=0` is exactly $(x^\top z)^2$ and finds the same support
vectors and intercept.

### 6.5 The Gaussian (RBF) kernel and its width γ

The most widely used kernel is the **radial basis function** (Gaussian) kernel

$$\kappa(x, z) = \exp\!\big(-\gamma\,\lVert x - z\rVert^2\big),$$

*In words:* similarity is 1 for identical samples and decays with squared distance; $\gamma$ sets how fast. Large
$\gamma$ means each support vector influences only its immediate neighbourhood (wiggly boundaries, risk of
overfitting); small $\gamma$ means smooth, almost linear boundaries. Its feature space is infinite-dimensional. γ is a
**hyper-parameter** that must be chosen — as is C. We use noisier rings and a large independent test set.

In [ ]:
# ── 6.5 RBF kernel: the effect of γ (C = 1) on noisy rings ──
X_nr, y_nr = make_circles(n_samples=150, noise=0.2, factor=0.5, random_state=SEED + 1)
X_nr_test, y_nr_test = make_circles(n_samples=2000, noise=0.2, factor=0.5, random_state=SEED + 2)
fig, axes = plt.subplots(1, 4, figsize=(12, 3.3))
for ax, gamma in zip(axes, [0.1, 1.0, 10.0, 300.0]):
    m = SVC(kernel="rbf", gamma=gamma, C=1.0).fit(X_nr, y_nr)
    plot_decision_regions(m.predict, X_nr, y_nr, ax=ax, pad=0.3, resolution=150,
                          title=f"γ = {gamma:g}: train {m.score(X_nr, y_nr):.2f}, test {m.score(X_nr_test, y_nr_test):.2f}\n"
                                f"{len(m.support_)} of {len(y_nr)} samples are SVs")
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("RBF-kernel SVM on noisy SIMULATED rings", fontsize=10); plt.tight_layout(); plt.show()

🔍 **What just happened?** With γ = 0.1 the kernel is so wide that the boundary is a large, overly smooth oval that
cannot hug the inner ring: training and test accuracy are both poor (0.71 / 0.73) — underfitting. γ = 1 draws a
sensible closed curve (0.86 / 0.89). As γ grows further the boundary starts to wrap around individual samples:
training accuracy keeps rising while test accuracy falls, and at γ = 300 the SVM fits 99 % of the training rings but
only 76 % of new ones, with every sample a support vector — it has memorised the training set. (Many support vectors
can mean either extreme: at γ = 0.1, with C = 1, all 150 samples are support vectors too, because the wide margin
contains them.)

**Why do SVMs work in huge feature spaces?** Slide 67 gives three reasons: the number of parameters (one $\alpha_i$
per sample) does not grow with the feature dimension; usually only a small group of support vectors matters; and
maximising the margin acts as a regulariser. The γ = 300 panel shows the honest caveat: the margin regularises only as
much as C and the kernel allow — **a kernel SVM can overfit, and its hyper-parameters must be selected on held-out
data.**

✏️ **Your turn.** Fix γ = 10 and vary C over 0.01, 1, 100, 10,000. Which values underfit, which overfit, and how does
the number of support vectors move?

---
## 7 · More than two classes: one-vs-all (slide 68)

> 📍 **Where we are.** The SVM is a two-class machine. Ramaswamy *et al.* had 14 tumour types (slide 2). The most
> common fix (slide 68) is **one-vs-all** (OvA): train one SVM per class, "this class vs everything else", and assign
> a new sample to the class whose SVM gives the largest score $w_k^\top x + b_k$.

We simulate three cell types measured by two markers.

In [ ]:
# ── 7.1 One-vs-all with three linear SVMs (SIMULATED cell types) ──
rng = np.random.default_rng(SEED + 7)
centres = np.array([[-2.0, -1.0], [2.0, -1.0], [0.0, 2.0]])
X_3 = np.vstack([rng.normal(c, 0.9, (40, 2)) for c in centres])
y_3 = np.repeat([0, 1, 2], 40)
models = [SVC(kernel="linear", C=1.0).fit(X_3, np.where(y_3 == k, 1, -1)) for k in range(3)]   # one per class


def ova_predict(Z):
    """Class whose 'k vs rest' SVM gives the largest score wₖᵀz + bₖ."""
    return np.argmax(np.column_stack([m.decision_function(Z) for m in models]), axis=1)


print("training accuracy (one-vs-all):", round(accuracy(y_3, ova_predict(X_3)), 3))
fig, ax = plt.subplots(figsize=(5, 4.2))
plot_decision_regions(ova_predict, X_3, y_3, ax=ax, pad=0.5, class_names=["type A", "type B", "type C"],
                      title="one-vs-all: three SVMs, highest score wins")
x1 = np.linspace(X_3[:, 0].min() - 0.5, X_3[:, 0].max() + 0.5, 2)
for k, m in enumerate(models):
    w_k, b_k = m.coef_[0], m.intercept_[0]
    ax.plot(x1, (-b_k - w_k[0] * x1) / w_k[1], ls=":", color=CLASS_COLORS[k], lw=1.5)
ax.set_xlabel("marker 1"); ax.set_ylabel("marker 2"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each dotted line is one "type vs rest" boundary. They do not meet neatly: some regions are
claimed by two classifiers and the central region by none. Taking the class with the **largest score** resolves both
cases and gives the shaded partition. (Scores of separately trained SVMs are not calibrated probabilities, which is a
known weakness of OvA.) Note that scikit-learn's `SVC` uses the other common scheme, **one-vs-one** (a classifier for
every pair of classes, then a vote), whereas `LinearSVC` uses one-vs-rest.

✏️ **Your turn.** Replace the three linear SVMs by RBF SVMs (`SVC(kernel="rbf", gamma=1.0, C=1.0)`) and redraw the
regions (drop the dotted lines, which need `coef_`). Then fit `SVC(kernel="linear").fit(X_3, y_3)` directly — that is
one-vs-one — and compare its training accuracy with our one-vs-all classifier.

### 📑 Slide connection — `007_SVM`, slides 57–68

- **What the slides show.** 1-D data that no threshold separates become separable after φ(x) = (x, x²) (58–60); 2-D data become separable after φ(x) = (x₁², x₂², √2x₁x₂) (61–62); the dual only needs φ(xᵢ)ᵀφ(xⱼ) = κ(xᵢ, xⱼ) (63); high-dimensional features are expensive and add parameters, but kernels and the dual solve both (64); the 6-D map and κ(x, x') = (γxᵀx' + r)^M with M = 2, γ = r = 1 can be computed without φ (65–66); why SVMs do not overfit (67); one-vs-all for multiclass (68).
- **What this notebook adds.** The 1-D example solved as a hard-margin SVM in (x, x²) with its two thresholds; both kernel identities verified numerically; timing of explicit features vs kernel as J grows; a polynomial-kernel SVM from our own dual solver, matched to scikit-learn; the RBF kernel and γ, including the overfitting regime that slide 67 glosses over; a one-vs-all classifier from three binary SVMs and the one-vs-one alternative.
- **Exam tip.** Be able to expand (xᵀz)² for 2-D x, z and recognise the feature map, to explain why the kernel trick needs the dual, and to name the hyper-parameters of polynomial and RBF kernels.

---
## 8 · Real data: leukaemia subtypes and honest model selection

> 📍 **Where we are.** We have all the machinery. Now the question every biologist using an SVM must answer:
> **which C, which kernel, which γ, which preprocessing?** The model-selection lecture lists, for each method, the
> knob that has to be chosen — and writes "SVM: NA" (`015_model-selection_mz_040725`, slide 3). That is an
> oversimplification. An SVM has no *structure* to select in the sense of a graph or a number of clusters, but it has
> hyper-parameters that change its predictions dramatically, as Sections 4 and 6 showed.

The **Golub et al. (1999)** data: bone-marrow and blood samples of 72 acute leukaemia patients, each profiled on
Affymetrix arrays (here 3,571 genes after the preprocessing of Dudoit *et al.* 2002, log10 scale). 47 have acute
lymphoblastic leukaemia (ALL) and 25 acute myeloid leukaemia (AML) — diseases that need different treatment. Golub's
original split is 38 training and 34 test samples. We code **AML = +1** and **ALL = −1**.

**The rules of the game.** Every choice — standardisation, C, kernel, γ — is made using the **38 training samples
only**, by leave-one-out cross-validation (LOOCV: fit on 37, predict the one left out, repeat 38 times). The scaler
is refit inside every fold (a scikit-learn *pipeline*), so that not even the gene means of the held-out sample leak
into training. The 34 test samples are used **once**, at the end.

In [ ]:
# ── 8.1 Load the Golub leukaemia data ──
golub = load_golub()
print("data source:", golub["source"])
X_g = golub["X"]
y_g = np.where(golub["y"] == 1, 1, -1)                          # +1 = AML, −1 = ALL
train_g = golub["is_train"]
X_gtr, y_gtr, X_gte, y_gte = X_g[train_g], y_g[train_g], X_g[~train_g], y_g[~train_g]
print(f"{X_g.shape[0]} patients × {X_g.shape[1]} genes | training: {len(y_gtr)} "
      f"({np.sum(y_gtr == 1)} AML, {np.sum(y_gtr == -1)} ALL) | test: {len(y_gte)} ({np.sum(y_gte == 1)} AML, {np.sum(y_gte == -1)} ALL)")

In [ ]:
# ── 8.2 Linear SVM: leave-one-out accuracy as a function of C ──
loo = LeaveOneOut()
C_grid = [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]
t0 = time.time()
linear_cv = {C: cross_val_score(make_pipeline(StandardScaler(), SVC(kernel="linear", C=C)), X_gtr, y_gtr, cv=loo).mean()
             for C in C_grid}
n_sv = {C: len(make_pipeline(StandardScaler(), SVC(kernel="linear", C=C)).fit(X_gtr, y_gtr)[-1].support_) for C in C_grid}
print(f"({time.time() - t0:.1f} s)")
table = pd.DataFrame({"C": C_grid, "LOOCV accuracy": [linear_cv[C] for C in C_grid],
                      "support vectors (of 38)": [n_sv[C] for C in C_grid]})
print(table.to_string(index=False, formatters={"C": "{:g}".format, "LOOCV accuracy": "{:.3f}".format}))
print(f"always predicting the majority class (ALL) would score {np.mean(y_gtr == -1):.3f}")

🔍 **What just happened?** For tiny C (≤ 10⁻⁴) the SVM is so heavily regularised that it barely moves away from
predicting the majority class: LOOCV accuracy 0.71–0.79, close to the 0.71 of "always ALL". From C = 10⁻³ on, 37 of
the 38 left-out patients are classified correctly (0.974), and the result no longer depends on C. Why? With 3,571 genes and 37 training patients the
two classes are **linearly separable** (in J ≫ N, almost any labelling is), so once C is large enough the soft-margin
SVM becomes the hard-margin SVM. Notice also that **28 of the 38 patients are support vectors** — slide 67's "usually
a small group of samples" is not true in the J ≫ N regime, where nearly every sample sits on the margin.

### 8.3 RBF kernel: a two-dimensional grid (C, γ)

In [ ]:
# ── 8.3 RBF kernel: LOOCV accuracy over a (γ, C) grid ──
gamma_grid = [1e-5, 1e-4, 1e-3, 1e-2]
C_rbf = [0.1, 1.0, 10.0, 100.0]
t0 = time.time()
rbf_cv = np.array([[cross_val_score(make_pipeline(StandardScaler(), SVC(kernel="rbf", C=C, gamma=g)),
                                    X_gtr, y_gtr, cv=loo).mean() for C in C_rbf] for g in gamma_grid])
print(f"({time.time() - t0:.1f} s); for reference, scikit-learn's default gamma='scale' is 1/(J·var) = 1/J "
      f"≈ {1 / X_gtr.shape[1]:.1e} here (var = 1 after standardisation)")

fig, ax = plt.subplots(figsize=(5.2, 3.6))
im = ax.imshow(rbf_cv, cmap="viridis", vmin=0.6, vmax=1.0)
ax.set_xticks(range(len(C_rbf)), [f"{c:g}" for c in C_rbf]); ax.set_yticks(range(len(gamma_grid)), [f"{g:g}" for g in gamma_grid])
for i in range(len(gamma_grid)):
    for j in range(len(C_rbf)):
        ax.text(j, i, f"{rbf_cv[i, j]:.2f}", ha="center", va="center", color="w" if rbf_cv[i, j] < 0.85 else "k", fontsize=8)
ax.set_xlabel("C"); ax.set_ylabel("γ"); ax.set_title("Golub training set (real): LOOCV accuracy, RBF SVM")
plt.colorbar(im, ax=ax, fraction=0.046); plt.tight_layout(); plt.show()

🔍 **What just happened?** The RBF SVM is very sensitive to γ. For γ ≥ 10⁻³ every cell of the grid shows 0.71 — the
SVM predicts "ALL" for everyone. In 3,571 standardised dimensions two patients are typically about
$\sqrt{2 \times 3571} \approx 85$ units apart, so $\exp(-\gamma\lVert x - z\rVert^2)$ is essentially 0 for every pair:
each patient is similar only to itself, and nothing generalises. Only for small γ (≈ 10⁻⁵–10⁻⁴, the scale of
1/J) and large enough C does the RBF SVM do well; it matches the linear SVM's 0.974 only at γ = 10⁻⁵, C = 100. **A bad hyper-parameter choice turns a
good classifier into a majority-class guesser** — that is why "SVM: NA" must be read as "no structure to select", not
"nothing to choose".

### 8.4 Choose, then test once

In [ ]:
# ── 8.4 Pick the best setting by LOOCV (ties → simpler model), then evaluate once on the 34 test patients ──
candidates = [(acc, 0, C, "linear", None) for C, acc in linear_cv.items()]                 # 0 = linear preferred on ties
candidates += [(rbf_cv[i, j], 1, C, "rbf", g) for i, g in enumerate(gamma_grid) for j, C in enumerate(C_rbf)]
best = max(candidates, key=lambda c: (c[0], -c[1], -c[2]))                                 # accuracy ↑, linear, small C
acc_cv, _, C_best, kernel_best, gamma_best = best
print(f"chosen on training data only: kernel={kernel_best}, C={C_best:g}, gamma={gamma_best}, LOOCV accuracy {acc_cv:.3f}")

params = dict(kernel=kernel_best, C=C_best) | ({"gamma": gamma_best} if kernel_best == "rbf" else {})
final = make_pipeline(StandardScaler(), SVC(**params)).fit(X_gtr, y_gtr)
pred = final.predict(X_gte)
cc = confusion_counts(y_gte == 1, pred == 1)                          # positive class = AML
print(f"TEST accuracy: {accuracy(y_gte, pred):.3f}  ({np.sum(pred != y_gte)} errors among {len(y_gte)} patients)")
print("confusion (AML = positive):", cc)
print("test patients misclassified (sample ids):", golub["sample_id"][~train_g][pred != y_gte])

🔍 **What just happened?** Several settings tie at the best LOOCV accuracy (0.974: the linear SVM for every
C ≥ 10⁻³ and the RBF SVM at γ = 10⁻⁵, C = 100); the tie-breaking rule picks the simplest, the linear SVM with
C = 10⁻³. On the untouched test set it misclassifies 2 of the 34 patients (accuracy 0.941) — both are AML patients
called ALL; no ALL patient is called AML. The LOOCV estimate was a little optimistic, as expected: choosing the best
of many settings on 38 samples introduces a selection bias (Module 23 discusses nested cross-validation as the
remedy), and Golub's test samples were collected more heterogeneously than the training samples (they include
peripheral-blood as well as bone-marrow samples) and contain a higher fraction of AML (14/34 vs 11/38). With 34 test
patients one error moves the accuracy by 3 percentage points, so never compare methods on such decimals.

### 8.5 Preprocessing is a model choice, too

In [ ]:
# ── 8.5 Feature scaling changes an SVM (Wisconsin breast cancer, real) ──
bc = load_breast_cancer()
X_bc, y_bc = bc.data, np.where(bc.target == 0, 1, -1)                 # +1 = malignant
rng = np.random.default_rng(SEED + 85)
tr_bc, te_bc = train_test_indices(len(y_bc), 0.3, rng, stratify=y_bc)
print("feature ranges differ by orders of magnitude, e.g. mean area", X_bc[:, 3].min().round(0), "–", X_bc[:, 3].max().round(0),
      "vs mean smoothness", X_bc[:, 4].min().round(3), "–", X_bc[:, 4].max().round(3))
for scaled in [False, True]:
    for kernel in ["linear", "rbf"]:
        model = SVC(kernel=kernel, C=1.0, gamma="scale")
        if scaled:
            model = make_pipeline(StandardScaler(), model)
        t0 = time.time(); model.fit(X_bc[tr_bc], y_bc[tr_bc])
        print(f"standardised={str(scaled):5s} kernel={kernel:6s}: test accuracy {model.score(X_bc[te_bc], y_bc[te_bc]):.3f}"
              f"  (fit {time.time() - t0:.2f} s)")

🔍 **What just happened?** SVMs measure geometry — margins, dot products, distances — so a feature measured in large
units (nuclear area, hundreds to thousands of pixels) dominates one measured in small units (smoothness, ~0.1).
Without standardisation both kernels lose accuracy (linear 0.947 vs 0.977, RBF 0.930 vs 0.971 on these 171 test
tumours) and the linear SVM takes much longer to fit (its optimisation problem is badly conditioned); after
standardisation both do well. Standardise with the **training** mean and standard
deviation only (the pipeline does this inside every cross-validation fold).

**Summary — the model-selection choices of an SVM:**

| choice | options | how to choose |
|---|---|---|
| cost C | $10^{-4}$ … $10^{4}$ (log grid) | cross-validation on training data |
| kernel | linear, polynomial (degree $M$, $\gamma$, $r$), RBF ($\gamma$) | cross-validation; prefer the simpler on ties |
| kernel parameters | γ on a log grid around 1/(J·var) | cross-validation |
| preprocessing | standardisation, feature selection (inside the CV loop!) | part of the pipeline |
| class weights | equal, or "balanced" for unequal class sizes (27 ALL vs 11 AML) | cross-validation, or the cost of each error type |
| multiclass scheme | one-vs-all, one-vs-one | convention / cross-validation |

✏️ **Your turn.** Repeat 8.2–8.4 with `SVC(..., class_weight="balanced")`, which gives each AML training patient more
weight. Does the chosen model or the test confusion matrix change? Why might a clinician care about the two error
types differently?

### 📑 Slide connection — `015_model-selection_mz_040725`, slides 3 (and 007_SVM slides 2–3)

- **What the slides show.** The model-selection lecture lists each method's model-selection knob — feature selection for regression, K for k-NN, tree size for decision trees, graph structure for Bayesian networks, number of clusters or states for mixtures and HMMs — and writes 'SVM: NA'. The SVM lecture opens with SVMs for cancer-type prediction from expression data (2–3).
- **What this notebook adds.** A complete, honest model selection on real leukaemia data: linear vs RBF kernel, C and γ grids, scaling refit inside leave-one-out folds, ties broken towards the simpler model, and a single final test (2 errors in 34). The heat map shows that a poor γ reduces the SVM to a majority-class guesser, and the breast-cancer experiment shows that preprocessing changes the result — so 'SVM: NA' should be read as 'no structure to select', not 'nothing to choose'.
- **Exam tip.** If asked what must be tuned in an SVM, answer: C, the kernel and its parameters (γ, degree), preprocessing and class weights — chosen by cross-validation on training data, never on the test set.

---
## 📌 Key takeaways

- The SVM is a **discriminative, non-probabilistic** linear classifier that chooses, among all separating
  boundaries, the one with the **maximum margin**; its probabilistic cousin is logistic regression.
- **Geometry:** $w$ is perpendicular to the boundary, $x^+ = x^- + \lambda w$ with $\lambda = 2/w^\top w$, so the margin
  is $M = 2/\lVert w\rVert$; maximising $M$ ⇔ minimising $\tfrac12 w^\top w$ subject to $y_i(w^\top x_i + b) \ge 1$ — a
  **quadratic program**.
- **Soft margin:** slacks $\varepsilon_i$ and cost $C$ give $\tfrac12 w^\top w + C\sum_i\varepsilon_i$ — equivalently ridge
  + **hinge loss** $[1 - t]_+$. Large C → narrow margin, few violations; small C → wide margin, many support vectors.
- **Dual:** $w = \sum_i \alpha_i y_i x_i$, $\sum_i\alpha_i y_i = 0$, $0 \le \alpha_i \le C$. Samples with $\alpha_i > 0$
  are the **support vectors** (on the margin if $0<\alpha_i<C$, inside or misclassified if $\alpha_i = C$); predictions
  need only dot products with them.
- **Kernel trick:** the dual uses only $x_i^\top x_j$; replacing it by $\kappa(x_i, x_j) = \phi(x_i)^\top\phi(x_j)$ —
  e.g. $(x^\top z)^2$, $(x^\top z + 1)^2$, $\exp(-\gamma\lVert x - z\rVert^2)$ — fits linear boundaries in huge feature
  spaces at the cost of the original dot product.
- **Multiclass:** one-vs-all (largest score wins) or one-vs-one.
- **SVMs need model selection:** C, kernel, γ, preprocessing and class weights, chosen by cross-validation on the
  training data. On the Golub leukaemia data a linear SVM chosen this way misclassifies 2 of 34 test patients; a badly
  chosen γ predicts the majority class for everyone.

---
## ✅ Checkpoint

1. An SVM has $w = (3, 4)$. What is the width of its margin?
   <details><summary>Answer</summary>$\lVert w\rVert = \sqrt{9 + 16} = 5$, so $M = 2/\lVert w\rVert = 0.4$ (in the units of $x$).</details>
2. Explain in one sentence why the margin planes can be fixed at $w^\top x + b = \pm1$.
   <details><summary>Answer</summary>Scaling $(w, b)$ by any $c > 0$ leaves the boundary unchanged, so we choose the scale at which the closest training points have score exactly ±1; then the margin is $2/\lVert w\rVert$.</details>
3. Write the five lines that derive $\lambda = 2/w^\top w$ from $w^\top x^+ + b = 1$, $w^\top x^- + b = -1$ and $x^+ = x^- + \lambda w$.
   <details><summary>Answer</summary>$w^\top x^+ + b = 1$ → $w^\top(x^- + \lambda w) + b = 1$ → $(w^\top x^- + b) + \lambda w^\top w = 1$ → $-1 + \lambda w^\top w = 1$ → $\lambda = 2/w^\top w$.</details>
4. A sample has score $t_i = y_i(w^\top x_i + b) = 0.4$ in a soft-margin SVM. What are its slack, its multiplier and its status?
   <details><summary>Answer</summary>$\varepsilon_i = 1 - 0.4 = 0.6$: it is inside the band but on the correct side. Its multiplier is at the cap, $\alpha_i = C$, so it is a support vector (a margin violator), not misclassified.</details>
5. What happens to the margin and the number of support vectors as C → 0? As C → ∞ on separable data?
   <details><summary>Answer</summary>C → 0: violations are free, the margin becomes very wide and (almost) all samples become support vectors. C → ∞ on separable data: the solution converges to the hard-margin SVM (narrowest margin with no violations, typically few support vectors).</details>
6. Expand $(x^\top z)^2$ for $x, z \in \mathbb{R}^2$ and read off the feature map.
   <details><summary>Answer</summary>$(x_1z_1 + x_2z_2)^2 = x_1^2z_1^2 + x_2^2z_2^2 + 2x_1x_2z_1z_2 = \phi(x)^\top\phi(z)$ with $\phi(x) = (x_1^2, x_2^2, \sqrt2\,x_1x_2)$.</details>
7. Read the code: in `svm_dual`, where does the hard-margin vs soft-margin difference appear, and how is w recovered afterwards?
   <details><summary>Answer</summary>Only in `bounds=[(0.0, C)] * N`: with `C=None` the multipliers are bounded below by 0 only (hard margin), otherwise by 0 and C (soft margin). Afterwards `w = (alpha * y) @ X` computes Σ αᵢ yᵢ xᵢ, and `intercept_from_alpha` averages yᵢ − Σⱼ αⱼ yⱼ K_ji over the margin support vectors.</details>
8. A colleague reports 100 % accuracy for an RBF SVM whose γ and C were chosen to maximise accuracy on the test set. What is wrong, and what should they do?
   <details><summary>Answer</summary>The test set was used for model selection, so its accuracy is optimistically biased. Choose C and γ by cross-validation within the training set (with preprocessing inside the folds), then evaluate once on the test set — or use nested cross-validation.</details>

## 🧠 Questions & discussion

1. Logistic regression gives probabilities, the SVM only a side of the boundary. In which clinical situations would you insist on probabilities? How could you get approximate probabilities from an SVM score (look up Platt scaling)?
2. On the Golub data 28 of 38 training patients are support vectors. What does this say about the geometry of J ≫ N data, and about slide 67's argument that 'only a small group of samples matters'?
3. The hinge loss ignores samples with t ≥ 1, the logistic loss never does. When could ignoring well-classified samples be an advantage, and when a disadvantage (think of mislabelled samples near vs far from the boundary)?
4. Kernels measure similarity. Which kernel would you design to compare DNA sequences, or protein structures, where a vector of numbers is not the natural representation? (Look up the spectrum kernel.)
5. Why must feature selection (e.g. picking the 50 most differentially expressed genes) happen *inside* the cross-validation loop? What would happen to the LOOCV estimate otherwise (Ambroise & McLachlan 2002)?

## 🚀 Bonus — going further

- Implement the **SMO** algorithm (sequential minimal optimisation; Platt 1998): repeatedly pick two multipliers $\alpha_i, \alpha_j$ and solve the dual exactly for them. Compare its speed with our SLSQP solver for N = 500.
- Use the **ν-SVM** parameterisation (`sklearn.svm.NuSVC`), where ν bounds the fraction of support vectors and margin errors; compare it with the C parameterisation on the rings data.
- **Recursive feature elimination** (SVM-RFE; Guyon et al. 2002): repeatedly drop the genes with the smallest $|w_j|$ of a linear SVM on the Golub training data, and track LOOCV accuracy as a function of the number of genes — with the elimination inside the CV loop.
- Compare the linear SVM with L2-regularised logistic regression (Module 09) on Golub and on the breast-cancer data across a grid of regularisation strengths: how similar are the decision functions?
- The repository's [GenerativeDiscriminativeClassification](../../GenerativeDiscriminativeClassification/) notebook compares naïve Bayes and logistic regression on breast-cancer data; add a linear and an RBF SVM to its comparison, and calibrate them with [ConformalPrediction](../../ConformalPrediction/).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 11 · Support vector machines: margins, support vectors and kernels
Input: SIMULATED two-marker tumours (separable and overlapping); SIMULATED rings and cell types;
       Golub leukaemia (72 × 3,571, real, 38/34 split); Wisconsin breast cancer (569 × 30, real)
Output: hard/soft-margin SVMs (primal and dual), support vectors, kernel SVMs, a selected Golub classifier
Method: quadratic programming (SciPy SLSQP), subgradient descent on the hinge loss, Lagrangian duality,
        kernel trick, leave-one-out cross-validation for C, kernel and γ
Key equation: M = 2/‖w‖;  min ½wᵀw + CΣεᵢ s.t. yᵢ(wᵀxᵢ + b) ≥ 1 − εᵢ;  w = Σαᵢyᵢxᵢ, 0 ≤ αᵢ ≤ C
Key parameters: C; kernel (linear / polynomial degree, γ, r / RBF γ); standardisation; class weights
What changed when I modified C / γ / the scaling: …
One failure mode: a large RBF γ (or unscaled features) makes every sample its own island → majority-class
                  predictions or memorisation
Connection to 02-620 slides: 007_SVM 2–13 (motivation, max margin), 16–31 (margin, QP), 32–41 & 69–73
                             (soft margin, hinge), 42–56 + extra (dual), 57–68 (kernels, multiclass);
                             015_model-selection slide 3 ("SVM: NA", corrected)
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists*, Spring 2025 lecture slides, decks `007_SVM`, `007_SVM_extra_slides` and `015_model-selection_mz_040725` (course design S. Kim; instructors M. Xu and M. Zhang).
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed., Sections 12.1–12.3 (support vector classifiers, kernels). Springer.
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*, Chapter 14 (kernels), in particular Section 14.5 (support vector machines). MIT Press. (The slides' reading list says 'Murphy Ch 4.5'; the SVM material is in Section 14.5.)
- Cortes C., Vapnik V. (1995) Support-vector networks. *Machine Learning* 20:273–297.
- Boser B. E., Guyon I. M., Vapnik V. N. (1992) A training algorithm for optimal margin classifiers. *Proceedings of the 5th Annual Workshop on Computational Learning Theory (COLT)*, 144–152.
- Ramaswamy S., Tamayo P., Rifkin R., *et al.* (2001) Multiclass cancer diagnosis using tumor gene expression signatures. *PNAS* 98:15149–15154.
- Huang S., Cai N., Pacheco P. P., Narrandes S., Wang Y., Xu W. (2018) Applications of support vector machine (SVM) learning in cancer genomics. *Cancer Genomics & Proteomics* 15:41–51.
- Golub T. R., Slonim D. K., Tamayo P., *et al.* (1999) Molecular classification of cancer: class discovery and class prediction by gene expression monitoring. *Science* 286:531–537.
- Dudoit S., Fridlyand J., Speed T. P. (2002) Comparison of discrimination methods for the classification of tumors using gene expression data. *Journal of the American Statistical Association* 97:77–87.
- Furey T. S., Cristianini N., Duffy N., Bednarski D. W., Schummer M., Haussler D. (2000) Support vector machine classification and validation of cancer tissue samples using microarray expression data. *Bioinformatics* 16:906–914.
- Ambroise C., McLachlan G. J. (2002) Selection bias in gene extraction on the basis of microarray gene-expression data. *PNAS* 99:6562–6566.
- Platt J. C. (1998) Sequential minimal optimization: a fast algorithm for training support vector machines. Microsoft Research Technical Report MSR-TR-98-14.
- Guyon I., Weston J., Barnhill S., Vapnik V. (2002) Gene selection for cancer classification using support vector machines. *Machine Learning* 46:389–422.
- Shalev-Shwartz S., Singer Y., Srebro N. (2007) Pegasos: primal estimated sub-gradient solver for SVM. *Proceedings of the 24th International Conference on Machine Learning (ICML)*, 807–814.
- Chang C.-C., Lin C.-J. (2011) LIBSVM: a library for support vector machines. *ACM Transactions on Intelligent Systems and Technology* 2:27 (the solver behind scikit-learn's `SVC`).

## ➡️ Next

Part B ends here: you can now fit, regularise and select supervised models. Part C drops the labels.
[Module 12](12_clustering_kmeans_hierarchical.ipynb) asks what structure a data set has when nobody tells us
the classes — clustering co-expressed genes with k-means and hierarchical clustering.